# Configurable Text Classification Pipeline — Banking77
### TF-IDF, Word2Vec, DistilBERT, classical classifiers, and bonus Word2Vec + BiLSTM

**Course:** Machine Learning ([Course code, e.g. CO3117]) - Semester I, Academic Year 2026-2027
**Faculty:** Computer Science and Engineering, Ho Chi Minh City University of Technology (HCMUT), VNU-HCM
**Instructor:** [Instructor full name and title] ([instructor email])
**Track:** Task 2 - Machine Learning with Text Data (+ bonus deep-learning pipeline)
**GitHub repository:** [https://github.com/your-username/your-repo](https://github.com/your-username/your-repo)

| Full name | Student ID | Email | Main responsibilities | Contribution |
| --- | --- | --- | --- | --- |
| [Member 1 full name] | [Student ID] | [member1@hcmut.edu.vn] | Data loading, boolean preprocessing, EDA, TF-IDF, Naive Bayes / LogReg / SVM, ablation, README | 50% |
| [Member 2 full name] | [Student ID] | [member2@hcmut.edu.vn] | Word2Vec / DistilBERT, .npy / .h5 persistence, bonus Word2Vec + BiLSTM, evaluation figures, report | 50% |

**What this notebook does** (Runtime -> Run all, no Google Drive; Banking77 downloaded from a public Hugging Face link):

| Section | Content |
| --- | --- |
| 0 | install missing libraries, recreate `modules/` if the notebook is opened alone |
| 1 | EDA on Banking77 (word counts, length distribution, word frequencies, class balance) |
| 2 | boolean preprocessing switches and one-flag-at-a-time ablation |
| 3 | feature extraction: TF-IDF, Word2Vec, DistilBERT, saved as `.npy` and `.h5` |
| 4 | classifiers: Naive Bayes, Logistic Regression, calibrated LinearSVC |
| 5 | bonus deep learning: Word2Vec + BiLSTM |
| 6 | comparison tables, calibration, confusion, error and negation analysis, conclusions |

All logic lives in `modules/`; the notebook calls the same code path as `python main.py`.

## 0. Setup

The first cell installs only missing libraries. The second finds `modules/`: repository checkout if present, otherwise clones `REPO_URL` or writes the embedded module files. A GPU runtime (T4) is recommended; CPU / MPS also work.

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "seaborn": "seaborn",
    "sklearn": "scikit-learn", "scipy": "scipy", "h5py": "h5py", "torch": "torch",
    "transformers": "transformers", "datasets": "datasets",
}
missing = [pip for module, pip in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", " ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("all required libraries are already installed")

In [ ]:
import os
from pathlib import Path

REPO_URL = ""  # e.g. "https://github.com/your-username/your-repo.git" once the repository is public
MODULE_SOURCES = {'__init__.py': '"""\nConfigurable text-classification pipeline (HCMUT Machine Learning assignment, Task 2).\n\n    config              PreprocessConfig, ExperimentConfig, dataset registry, hyperparameters\n    data_loader         Hugging Face download + boolean preprocessing\n    eda                 word-count statistics, length distribution, word frequencies\n    feature_extraction  TF-IDF / Word2Vec / DistilBERT (.npy + .h5); BoW available as plug-in\n    classifiers         Naive Bayes, Logistic Regression, calibrated LinearSVC\n    deep_learning       Word2Vec + BiLSTM (bonus)\n    evaluation          ablation + comparison suites, CSV tables, figures\n\nHeavy libraries (torch, transformers) are imported lazily inside the functions that need them.\n"""\n\nfrom modules.config import (\n    DATASET_ORDER,\n    DATASET_REGISTRY,\n    FEATURE_DIR,\n    FIG_DIR,\n    REPORT_DIR,\n    SEED,\n    ExperimentConfig,\n    PreprocessConfig,\n)\nfrom modules.data_loader import clean_text, load_and_preprocess_dataset, load_raw_dataset\nfrom modules.eda import run_eda\nfrom modules.feature_extraction import extract_and_save_features, list_feature_files, load_h5_features\nfrom modules.classifiers import compute_metrics, train_and_evaluate_classifier\nfrom modules.deep_learning import run_bilstm_pipeline, run_dl_pipeline\nfrom modules.evaluation import (\n    build_master_table,\n    error_analysis,\n    negation_analysis,\n    plot_suite_results,\n    run_all_eda,\n    run_dataset_benchmark_suite,\n    run_deep_learning,\n    run_feature_extraction,\n    run_main_comparison,\n    run_preprocessing_ablation,\n)\nfrom modules.utils import get_device, seed_everything\n\n__all__ = [\n    "DATASET_ORDER", "DATASET_REGISTRY", "FEATURE_DIR", "FIG_DIR", "REPORT_DIR", "SEED",\n    "ExperimentConfig", "PreprocessConfig",\n    "clean_text", "load_and_preprocess_dataset", "load_raw_dataset",\n    "run_eda",\n    "extract_and_save_features", "list_feature_files", "load_h5_features",\n    "compute_metrics", "train_and_evaluate_classifier",\n    "run_bilstm_pipeline", "run_dl_pipeline",\n    "build_master_table", "error_analysis", "negation_analysis", "plot_suite_results", "run_all_eda",\n    "run_dataset_benchmark_suite", "run_deep_learning", "run_feature_extraction", "run_main_comparison",\n    "run_preprocessing_ablation",\n    "get_device", "seed_everything",\n]\n', 'classifiers.py': '"""\nDownstream classifiers on saved features, with one shared metric dictionary.\n\nclassifier_type   sparse input (bow / tfidf)        dense input (embeddings)\n---------------   -------------------------------   --------------------------------\nnaive_bayes       MultinomialNB(alpha=0.5)          GaussianNB()\nlogreg            LogisticRegression(C=2.0)          StandardScaler + LogisticRegression(C=2.0)\nsvm               Calibrated LinearSVC(C=1.0)        L2 Normalizer + Calibrated LinearSVC(C=1.0)\n"""\n\nfrom __future__ import annotations\n\nimport time\n\nimport numpy as np\nfrom scipy import sparse\nfrom sklearn.calibration import CalibratedClassifierCV\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score\nfrom sklearn.naive_bayes import GaussianNB, MultinomialNB\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import Normalizer, StandardScaler\nfrom sklearn.svm import LinearSVC\n\nfrom modules.config import SEED\n\nCLASSIFIER_NAMES = {\n    "naive_bayes": "Naive Bayes",\n    "logreg": "LogReg",\n    "svm": "LinearSVC (calibrated)",\n}\n\n\ndef multiclass_brier(y_true, proba) -> float:\n    """Mean over samples of sum_k (p_k - 1[y = k])^2; 0 is perfect, 2 is worst."""\n    onehot = np.zeros_like(proba)\n    onehot[np.arange(len(y_true)), y_true] = 1.0\n    return float(np.mean(np.sum((proba - onehot) ** 2, axis=1)))\n\n\ndef expected_calibration_error(y_true, proba, n_bins: int = 15) -> float:\n    """Top-label ECE: |accuracy - confidence| averaged over equal-width confidence bins."""\n    conf = proba.max(axis=1)\n    correct = (proba.argmax(axis=1) == y_true).astype(float)\n    edges = np.linspace(0, 1, n_bins + 1)\n    ece = 0.0\n    for lo, hi in zip(edges[:-1], edges[1:]):\n        sel = (conf > lo) & (conf <= hi)\n        if sel.any():\n            ece += sel.mean() * abs(correct[sel].mean() - conf[sel].mean())\n    return float(ece)\n\n\ndef compute_metrics(y_true, proba, method: str, train_time: float = 0.0, infer_time: float = 0.0) -> dict:\n    y_true = np.asarray(y_true)\n    proba = np.clip(np.asarray(proba, dtype=np.float64), 0.0, 1.0)\n    proba = proba / proba.sum(axis=1, keepdims=True).clip(min=1e-12)\n    pred = proba.argmax(axis=1)\n    metrics = {\n        "method": method,\n        "accuracy": float(accuracy_score(y_true, pred)),\n        "precision": float(precision_score(y_true, pred, average="macro", zero_division=0)),\n        "recall": float(recall_score(y_true, pred, average="macro", zero_division=0)),\n        "f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),\n        "brier_score": multiclass_brier(y_true, proba),\n        "ece": expected_calibration_error(y_true, proba),\n        "train_time_sec": float(train_time),\n        "infer_time_sec": float(infer_time),\n        "predictions": pred,\n        "probabilities": proba.astype(np.float32),\n    }\n    print(\n        f"  {method:42s} acc={metrics[\'accuracy\']:.4f} F1={metrics[\'f1\']:.4f} "\n        f"Brier={metrics[\'brier_score\']:.4f} ECE={metrics[\'ece\']:.4f} "\n        f"train={train_time:.1f}s infer={infer_time:.2f}s"\n    )\n    return metrics\n\n\ndef align_proba(model_classes, proba, n_classes):\n    out = np.zeros((proba.shape[0], n_classes), dtype=np.float64)\n    out[:, np.asarray(model_classes, dtype=int)] = proba\n    return out\n\n\ndef build_classifier(classifier_type: str, is_sparse: bool, seed: int = SEED, n_classes: int = 2):\n    if classifier_type == "naive_bayes":\n        return MultinomialNB(alpha=0.5) if is_sparse else GaussianNB()\n    if classifier_type == "logreg":\n        clf = LogisticRegression(\n            C=2.0, solver="lbfgs", max_iter=2000, class_weight="balanced", random_state=seed\n        )\n        return clf if is_sparse else make_pipeline(StandardScaler(), clf)\n    if classifier_type == "svm":\n        svc = LinearSVC(C=1.0, class_weight="balanced", dual="auto", max_iter=5000, random_state=seed)\n        clf = CalibratedClassifierCV(svc, cv=3, method="sigmoid")\n        # Dense SVM: L2-normalize rows for stable, fast convergence.\n        return clf if is_sparse else make_pipeline(Normalizer(), clf)\n    raise ValueError(f"Unknown classifier_type \'{classifier_type}\'")\n\n\ndef train_and_evaluate_classifier(\n    features_dict: dict,\n    y_train: np.ndarray,\n    y_test: np.ndarray,\n    classifier_type: str,\n    method_label: str,\n    seed: int = SEED,\n    use_projection: bool = False,\n    n_classes: int | None = None,\n) -> dict:\n    """\n    Fit one classifier on saved features and return the shared metric dictionary.\n\n    Set `use_projection=True` to run on the 30-d PCA/SVD projection (optional control).\n    """\n    n_classes = n_classes or int(max(y_train.max(), y_test.max()) + 1)\n    reduced = use_projection\n    x_train = features_dict["X_train_pca30"] if reduced else features_dict["X_train"]\n    x_test = features_dict["X_test_pca30"] if reduced else features_dict["X_test"]\n    is_sparse = sparse.issparse(x_train)\n\n    model = build_classifier(classifier_type, is_sparse, seed, n_classes=n_classes)\n    start = time.perf_counter()\n    model.fit(x_train, y_train)\n    train_time = time.perf_counter() - start\n    start = time.perf_counter()\n    proba = align_proba(model.classes_, model.predict_proba(x_test), n_classes)\n    infer_time = time.perf_counter() - start\n\n    metrics = compute_metrics(y_test, proba, method_label, train_time, infer_time)\n    metrics["classifier"] = classifier_type\n    metrics["input"] = "30-d projection" if reduced else ("sparse" if is_sparse else "dense")\n    metrics["input_dim"] = int(np.asarray(x_train.shape[1]))\n    return metrics\n', 'config.py': '"""\nPipeline configuration: dataset, boolean preprocessing, experiment knobs.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nSEED = 42\n\n# Assignment folder layout: notebooks/, modules/, features/, reports/.\n# Smoke tests (TC_QUICK=1) write to .quick/ so they never overwrite real results.\nROOT = Path(__file__).resolve().parents[1]\n_OUT = ROOT / ".quick" if os.environ.get("TC_QUICK", "0") == "1" else ROOT\nFEATURE_DIR = _OUT / "features"\nREPORT_DIR = _OUT / "reports"\nFIG_DIR = REPORT_DIR / "figures"\n\nDATASET_REGISTRY = {\n    "banking77": {\n        "title": "Banking77",\n        "hf_id": "mteb/banking77",\n        "max_train": None,\n        "max_test": None,\n        "max_len_cap": 64,\n    },\n}\n\nDATASET_ORDER = ["banking77"]\n\n# Sparse lexical features\nMAX_FEATURES = 25000\n\n# Dense projection for PCA/SVD-30 (saved with features; not used in the default grid)\nPCA_DIM = 30\n\n# Frozen Transformer encoder\nBERT_MODEL = "distilbert-base-uncased"\nBERT_BATCH_SIZE = 32\n\n# Word2Vec (train on the train split; optional KEYEDVECTORS override)\nWORD2VEC = {\n    "vector_size": 300,\n    "window": 5,\n    "min_count": 2,\n    "epochs": 5,\n    "sg": 1,  # skip-gram\n    "workers": 4,\n    "pretrained_env": "WORD2VEC_KV_PATH",\n}\n\n# Bonus sequence model\nBILSTM = {\n    "max_vocab": 20000,\n    "embed_dim": 128,  # used only for random-init fallback\n    "hidden_dim": 192,\n    "dropout": 0.3,\n    "batch_size": 64,\n    "epochs": 7,\n    "lr": 1e-3,\n    "patience": 2,\n    "max_len_cap": 200,\n}\n\nDL_EMBEDDINGS = ("word2vec",)\nDL_MODELS = ("bilstm",)\n\n\n@dataclass\nclass PreprocessConfig:\n    """Boolean preprocessing switches required by the assignment."""\n\n    stop_word_removal: bool = False\n    tokenization: bool = True\n    padding: bool = True\n    negation_marking: bool = False\n\n    def tag(self) -> str:\n        return (\n            f"sw{int(self.stop_word_removal)}_"\n            f"tok{int(self.tokenization)}_"\n            f"pad{int(self.padding)}_"\n            f"neg{int(self.negation_marking)}"\n        )\n\n\n@dataclass\nclass ExperimentConfig:\n    dataset_name: str = "banking77"\n    preprocess: PreprocessConfig = field(default_factory=PreprocessConfig)\n    embedding_type: str = "tfidf"  # bow | tfidf | word2vec | distilbert\n    classifier_type: str = "logreg"  # naive_bayes | logreg | svm\n    dl_model: str = "bilstm"\n    dl_embedding: str = "word2vec"\n    save_format: str = "both"  # npy | h5 | both\n\n\nDEFAULT_PREPROCESS = PreprocessConfig(\n    stop_word_removal=False,\n    tokenization=True,\n    padding=True,\n    negation_marking=False,\n)\n', 'data_loader.py': '"""\nDataset loading and boolean text preprocessing.\n\nEvery dataset is returned in the same dictionary shape, so the feature,\nclassifier and deep-learning steps never need to know which corpus they run on:\n\n    {\n        "name", "title", "label_names", "n_classes", "max_len_cap",\n        "train_raw", "test_raw",          # untouched strings (for error analysis)\n        "train_text", "test_text",        # strings after clean_text(prep_cfg)\n        "train_label", "test_label",      # np.int64 arrays\n        "p95_len",                        # 95th percentile words per train document\n        "preprocess_tag",                 # e.g. "sw0_tok1_pad1_neg0"\n    }\n\nAll corpora are downloaded from public Hugging Face links; nothing is read from\nGoogle Drive.\n"""\n\nfrom __future__ import annotations\n\nimport html\nimport os\nimport re\nfrom functools import lru_cache\n\nimport numpy as np\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS\nfrom sklearn.model_selection import train_test_split\n\nfrom modules.config import DATASET_REGISTRY, SEED, PreprocessConfig\n\n_URL = re.compile(r"https?://\\S+|www\\.\\S+")\n_HTML_TAG = re.compile(r"<[^>]+>")\n_HTML_ENTITY_LEFTOVER = re.compile(r"#\\d+;|&[a-z]+;")\n_BACKSLASH = re.compile(r"\\\\+")\n_SPACE = re.compile(r"\\s+")\n# Words (with an optional contraction tail such as "n\'t" or "\'s") or single punctuation marks.\n_TOKEN = re.compile(r"[a-z0-9]+(?:\'[a-z]+)?|[^\\sa-z0-9]")\n\nSTOP_WORDS = frozenset(ENGLISH_STOP_WORDS)\n# Cue tokens that attach to the following alphanumeric word when negation_marking=True.\nNEGATION_CUES = frozenset({\n    "not", "no", "never", "none", "nobody", "nothing", "nowhere", "neither",\n    "nor", "cannot", "without", "n\'t",\n})\n\n\ndef quick_mode() -> bool:\n    """`TC_QUICK=1` shrinks every split so the whole suite runs in minutes (smoke tests)."""\n    return os.environ.get("TC_QUICK", "0") == "1"\n\n\ndef normalize_text(text: str) -> str:\n    """Lowercase, unescape HTML, drop URLs/tags/backslash artefacts, collapse whitespace."""\n    text = html.unescape(str(text)).lower()\n    text = _URL.sub(" ", text)\n    text = _HTML_TAG.sub(" ", text)\n    text = _HTML_ENTITY_LEFTOVER.sub(" ", text)\n    text = _BACKSLASH.sub(" ", text)\n    return _SPACE.sub(" ", text).strip()\n\n\ndef apply_negation_marking(tokens: list[str]) -> list[str]:\n    """Join negation cues with the next alphanumeric token: not good -> not_good."""\n    out: list[str] = []\n    i = 0\n    while i < len(tokens):\n        tok = tokens[i]\n        cue = tok.lstrip("\'") if tok.startswith("n\'") else tok\n        if cue in NEGATION_CUES or tok.endswith("n\'t"):\n            # Prefer attaching to the next alphanumeric content word.\n            j = i + 1\n            while j < len(tokens) and not tokens[j][0].isalnum():\n                out.append(tokens[j])\n                j += 1\n            if j < len(tokens) and tokens[j][0].isalnum():\n                prefix = "not" if tok.endswith("n\'t") else cue\n                out.append(f"{prefix}_{tokens[j]}")\n                i = j + 1\n                continue\n        out.append(tok)\n        i += 1\n    return out\n\n\ndef clean_text(\n    text: str,\n    stop_word_removal: bool = False,\n    keep_punctuation: bool = True,\n    negation_marking: bool = False,\n) -> str:\n    """\n    Clean one document.\n\n    Negation marking (if enabled) runs after tokenization and *before* stop-word\n    removal so cues like "not" are not dropped before they can attach. Punctuation\n    is kept by default because Transformer encoders use it to scope negation and\n    clause boundaries. A document is never returned empty.\n    """\n    base = normalize_text(text)\n    needs_tokenize = stop_word_removal or negation_marking or not keep_punctuation\n    if not needs_tokenize:\n        return base\n\n    tokens = _TOKEN.findall(base)\n    if negation_marking:\n        tokens = apply_negation_marking(tokens)\n    if not keep_punctuation:\n        tokens = [tok for tok in tokens if tok[0].isalnum() or "_" in tok]\n    if stop_word_removal:\n        # Keep marked phrases (not_good); drop plain stop words only.\n        tokens = [tok for tok in tokens if "_" in tok or tok not in STOP_WORDS]\n    cleaned = " ".join(tokens)\n    if cleaned.strip():\n        return cleaned\n    fallback = " ".join(tok for tok in _TOKEN.findall(base) if tok[0].isalnum())\n    return fallback or base or "empty"\n\n\ndef clean_corpus(\n    texts,\n    stop_word_removal: bool = False,\n    keep_punctuation: bool = True,\n    negation_marking: bool = False,\n) -> list[str]:\n    return [\n        clean_text(t, stop_word_removal, keep_punctuation, negation_marking)\n        for t in texts\n    ]\n\n\ndef word_counts(texts) -> np.ndarray:\n    return np.array([len(str(t).split()) for t in texts], dtype=np.int32)\n\n\ndef _stratified_cap(texts, labels, n, seed):\n    texts = [str(t) for t in texts]\n    labels = np.asarray(labels, dtype=np.int64)\n    if n is None or len(texts) <= n:\n        return texts, labels\n    index, _ = train_test_split(\n        np.arange(len(texts)), train_size=n, stratify=labels, random_state=seed\n    )\n    index = np.sort(index)\n    return [texts[i] for i in index], labels[index]\n\n\ndef _names_from_label_text(split) -> list[str]:\n    pairs = sorted({(int(lab), name) for lab, name in zip(split["label"], split["label_text"])})\n    return [name for _, name in pairs]\n\n\n@lru_cache(maxsize=None)\ndef _download(name: str):\n    """Download one dataset from the Hugging Face Hub (cached per process)."""\n    from datasets import load_dataset\n\n    if name not in DATASET_REGISTRY:\n        raise KeyError(f"Unknown dataset \'{name}\'. Choose one of: {\', \'.join(DATASET_REGISTRY)}")\n    raw = load_dataset(DATASET_REGISTRY[name]["hf_id"])\n    train, test = raw["train"], raw["test"]\n    if "label_text" in train.column_names:\n        names = _names_from_label_text(train)\n    else:\n        names = list(train.features["label"].names)\n    return (\n        tuple(names),\n        tuple(train["text"]),\n        tuple(int(x) for x in train["label"]),\n        tuple(test["text"]),\n        tuple(int(x) for x in test["label"]),\n    )\n\n\ndef load_raw_dataset(name: str, seed: int = SEED) -> dict:\n    """Official train/test split after stratified sub-sampling, before any cleaning."""\n    cfg = DATASET_REGISTRY[name]\n    names, tr_text, tr_lab, te_text, te_lab = _download(name)\n    max_train, max_test = cfg["max_train"], cfg["max_test"]\n    if quick_mode():\n        max_train = min(max_train or 10**9, 1500)\n        max_test = min(max_test or 10**9, 600)\n    tr_text, tr_lab = _stratified_cap(tr_text, tr_lab, max_train, seed)\n    te_text, te_lab = _stratified_cap(te_text, te_lab, max_test, seed + 1)\n\n    def _drop_blank(texts, labels):\n        keep = [i for i, t in enumerate(texts) if normalize_text(t)]\n        return [texts[i] for i in keep], labels[keep]\n\n    tr_text, tr_lab = _drop_blank(tr_text, tr_lab)\n    te_text, te_lab = _drop_blank(te_text, te_lab)\n    return {\n        "name": name,\n        "title": cfg["title"],\n        "hf_id": cfg["hf_id"],\n        "label_names": list(names),\n        "n_classes": len(names),\n        "max_len_cap": int(cfg["max_len_cap"]),\n        "train_raw": tr_text,\n        "test_raw": te_text,\n        "train_label": tr_lab,\n        "test_label": te_lab,\n    }\n\n\ndef load_and_preprocess_dataset(\n    dataset_name: str, prep_cfg: PreprocessConfig | None = None, seed: int = SEED\n) -> dict:\n    """Load one dataset and apply the boolean preprocessing configuration."""\n    prep_cfg = prep_cfg or PreprocessConfig()\n    data = load_raw_dataset(dataset_name, seed)\n    data["train_text"] = clean_corpus(\n        data["train_raw"], prep_cfg.stop_word_removal, True, prep_cfg.negation_marking\n    )\n    data["test_text"] = clean_corpus(\n        data["test_raw"], prep_cfg.stop_word_removal, True, prep_cfg.negation_marking\n    )\n    data["p95_len"] = int(np.ceil(np.percentile(word_counts(data["train_text"]), 95)))\n    data["preprocess_tag"] = prep_cfg.tag()\n    data["preprocess"] = prep_cfg\n    print(\n        f"[data] {data[\'title\']} ({prep_cfg.tag()}): {len(data[\'train_text\'])} train / "\n        f"{len(data[\'test_text\'])} test, {data[\'n_classes\']} classes, p95 words = {data[\'p95_len\']}"\n    )\n    return data\n', 'deep_learning.py': '"""\nBonus deep-learning pipeline: Word2Vec + BiLSTM for sequence modelling.\n"""\n\nfrom __future__ import annotations\n\nimport time\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\n\nfrom modules.classifiers import compute_metrics\nfrom modules.config import BILSTM, DL_EMBEDDINGS, DL_MODELS, FEATURE_DIR, SEED, ExperimentConfig, PreprocessConfig\nfrom modules.data_loader import quick_mode\nfrom modules.feature_extraction import cache_dir_for, extract_and_save_features, train_or_load_word2vec\nfrom modules.utils import get_device\n\nUNPADDED_MAX_LEN = 10\n\n\ndef _resolve_max_len(lengths, prep: PreprocessConfig, cap: int) -> int:\n    if not prep.padding:\n        return UNPADDED_MAX_LEN\n    p95 = int(np.ceil(np.percentile(lengths, 95))) if len(lengths) else 20\n    return int(min(cap, max(20, p95)))\n\n\ndef _pad_ids(id_lists, max_len, pad_id=0):\n    mat = np.full((len(id_lists), max_len), pad_id, dtype=np.int64)\n    lengths = np.ones(len(id_lists), dtype=np.int64)\n    for i, ids in enumerate(id_lists):\n        ids = ids[:max_len] or [1]\n        mat[i, : len(ids)] = ids\n        lengths[i] = len(ids)\n    return mat, lengths\n\n\ndef _word2vec_matrices(texts, kv, max_len):\n    word2idx = {w: i + 2 for i, w in enumerate(kv.index_to_key)}\n    id_lists = [[word2idx.get(w, 1) for w in t.split()] for t in texts]\n    return _pad_ids(id_lists, max_len, pad_id=0), word2idx\n\n\ndef _embedding_matrix_from_kv(kv, word2idx):\n    dim = kv.vector_size\n    weight = np.zeros((len(word2idx) + 2, dim), dtype=np.float32)\n    for w, i in word2idx.items():\n        weight[i] = kv[w]\n    rng = np.random.default_rng(SEED)\n    weight[1] = rng.normal(0, 0.01, dim).astype(np.float32)  # UNK\n    return weight\n\n\ndef _build_bilstm(input_dim: int, n_classes: int, pack: bool, embed_weight=None):\n    import torch\n    from torch import nn\n    from torch.nn.utils.rnn import pack_padded_sequence\n\n    class BiLSTMClassifier(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.pack = pack\n            self.use_embed = embed_weight is not None\n            if self.use_embed:\n                self.embedding = nn.Embedding.from_pretrained(\n                    torch.tensor(embed_weight), freeze=False, padding_idx=0\n                )\n                in_dim = embed_weight.shape[1]\n            else:\n                self.embedding = None\n                in_dim = input_dim\n            self.lstm = nn.LSTM(\n                in_dim, BILSTM["hidden_dim"], num_layers=1, batch_first=True, bidirectional=True\n            )\n            self.dropout = nn.Dropout(BILSTM["dropout"])\n            self.fc = nn.Linear(2 * BILSTM["hidden_dim"], n_classes)\n\n        def forward(self, x, lengths, mask=None):\n            emb = self.dropout(self.embedding(x)) if self.use_embed else self.dropout(x)\n            if self.pack:\n                emb = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)\n            _, (hidden, _) = self.lstm(emb)\n            joined = torch.cat([hidden[-2], hidden[-1]], dim=1)\n            return self.fc(self.dropout(joined))\n\n    return BiLSTMClassifier()\n\n\ndef _train_bilstm(model, train_batches, val_batches, test_batches, y_fit, y_val, y_test,\n                  n_classes, epochs, lr, patience, batch_size, seed, device, method_label):\n    import torch\n    from torch import nn\n    from torch.utils.data import DataLoader, TensorDataset\n\n    torch.manual_seed(seed)\n    model = model.to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=lr)\n    loss_fn = nn.CrossEntropyLoss()\n\n    def make_loader(batches, labels, shuffle):\n        ids, lengths = batches\n        ds = TensorDataset(\n            torch.from_numpy(ids),\n            torch.from_numpy(lengths),\n            torch.from_numpy(np.asarray(labels, dtype=np.int64)),\n        )\n        gen = torch.Generator().manual_seed(seed)\n        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=gen)\n\n    fit_loader = make_loader(train_batches, y_fit, True)\n    val_loader = make_loader(val_batches, y_val, False)\n    test_loader = make_loader(test_batches, y_test, False)\n\n    def run_epoch(dl, train):\n        model.train(train)\n        total, correct, count = 0.0, 0, 0\n        with torch.set_grad_enabled(train):\n            for ids, lengths, yb in dl:\n                ids, yb = ids.to(device), yb.to(device)\n                logits = model(ids, lengths)\n                loss = loss_fn(logits, yb)\n                if train:\n                    optimizer.zero_grad()\n                    loss.backward()\n                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                    optimizer.step()\n                total += loss.item() * len(yb)\n                correct += (logits.argmax(1) == yb).sum().item()\n                count += len(yb)\n        return total / max(count, 1), correct / max(count, 1)\n\n    history, best_state, best_val, wait = [], None, float("inf"), 0\n    start = time.perf_counter()\n    for epoch in range(1, epochs + 1):\n        tr_loss, tr_acc = run_epoch(fit_loader, True)\n        va_loss, va_acc = run_epoch(val_loader, False)\n        history.append({\n            "epoch": epoch,\n            "train_loss": tr_loss,\n            "train_acc": tr_acc,\n            "val_loss": va_loss,\n            "val_acc": va_acc,\n        })\n        print(\n            f"  epoch {epoch}/{epochs} train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} "\n            f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}"\n        )\n        if va_loss < best_val:\n            best_val, wait = va_loss, 0\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            wait += 1\n            if wait >= patience:\n                print("  early stopping")\n                break\n    train_time = time.perf_counter() - start\n    model.load_state_dict(best_state)\n    model.eval()\n    start = time.perf_counter()\n    probs = []\n    with torch.no_grad():\n        for ids, lengths, _ in test_loader:\n            logits = model(ids.to(device), lengths)\n            probs.append(torch.softmax(logits, dim=1).cpu().numpy())\n    infer_time = time.perf_counter() - start\n    metrics = compute_metrics(y_test, np.concatenate(probs), method_label, train_time, infer_time)\n    metrics.update({\n        "history": history,\n        "epochs_run": len(history),\n        "n_params": sum(p.numel() for p in model.parameters()),\n    })\n    return metrics\n\n\ndef _prepare_word2vec_sequences(dataset_dict, prep: PreprocessConfig, max_len_cap: int):\n    from sklearn.model_selection import train_test_split\n\n    y_train = dataset_dict["train_label"]\n    fit_idx, val_idx = train_test_split(\n        np.arange(len(y_train)), test_size=0.1, stratify=y_train, random_state=SEED\n    )\n    fit_text = [dataset_dict["train_text"][i] for i in fit_idx]\n    val_text = [dataset_dict["train_text"][i] for i in val_idx]\n    test_text = dataset_dict["test_text"]\n\n    extract_and_save_features(\n        dataset_dict, ExperimentConfig(dataset_dict["name"], prep, "word2vec")\n    )\n    folder = cache_dir_for(dataset_dict["name"], "word2vec", prep.tag(), FEATURE_DIR)\n    kv = train_or_load_word2vec(dataset_dict["train_text"], folder, SEED)\n    lengths = [len(t.split()) for t in fit_text]\n    max_len = _resolve_max_len(lengths, prep, max_len_cap)\n    (fit_ids, fit_len), word2idx = _word2vec_matrices(fit_text, kv, max_len)\n    (val_ids, val_len), _ = _word2vec_matrices(val_text, kv, max_len)\n    (test_ids, test_len), _ = _word2vec_matrices(test_text, kv, max_len)\n    weight = _embedding_matrix_from_kv(kv, word2idx)\n    return {\n        "fit": (fit_ids, fit_len),\n        "val": (val_ids, val_len),\n        "test": (test_ids, test_len),\n        "y_fit": y_train[fit_idx],\n        "y_val": y_train[val_idx],\n        "embed_weight": weight,\n        "input_dim": weight.shape[1],\n        "max_len": max_len,\n        "device": get_device(),\n    }\n\n\ndef run_bilstm_pipeline(\n    dataset_dict: dict,\n    prep_cfg: PreprocessConfig | None = None,\n    dl_embedding: str = "word2vec",\n    epochs: int | None = None,\n    seed: int = SEED,\n    loss_plot_path: Path | None = None,\n    show: bool = False,\n) -> dict:\n    if dl_embedding != "word2vec":\n        raise ValueError("This bonus BiLSTM path supports only Word2Vec embeddings")\n    prep_cfg = prep_cfg or dataset_dict.get("preprocess") or PreprocessConfig()\n    epochs = epochs or (2 if quick_mode() else BILSTM["epochs"])\n    data = _prepare_word2vec_sequences(dataset_dict, prep_cfg, BILSTM["max_len_cap"])\n    model = _build_bilstm(\n        data["input_dim"],\n        dataset_dict["n_classes"],\n        pack=prep_cfg.padding,\n        embed_weight=data["embed_weight"],\n    )\n    label = "BiLSTM (word2vec)"\n    print(\n        f"[bilstm] {label} {prep_cfg.tag()} on {data[\'device\']}: "\n        f"input_dim={data[\'input_dim\']}, max_len={data[\'max_len\']}"\n    )\n    metrics = _train_bilstm(\n        model,\n        data["fit"],\n        data["val"],\n        data["test"],\n        data["y_fit"],\n        data["y_val"],\n        dataset_dict["test_label"],\n        dataset_dict["n_classes"],\n        epochs,\n        BILSTM["lr"],\n        BILSTM["patience"],\n        BILSTM["batch_size"],\n        seed,\n        data["device"],\n        label,\n    )\n    metrics.update({\n        "classifier": "bilstm",\n        "input": dl_embedding,\n        "input_dim": data["max_len"],\n        "max_len": data["max_len"],\n        "vocab_size": int(data["embed_weight"].shape[0]),\n        "unk_rate": 0.0,\n        "dl_embedding": dl_embedding,\n    })\n    if loss_plot_path is not None and metrics.get("history"):\n        history = metrics["history"]\n        epochs_axis = [h["epoch"] for h in history]\n        fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))\n        axes[0].plot(epochs_axis, [h["train_loss"] for h in history], "o-", label="train")\n        axes[0].plot(epochs_axis, [h["val_loss"] for h in history], "o-", label="validation")\n        axes[0].set_title("Cross-entropy loss")\n        axes[1].plot(epochs_axis, [h["train_acc"] for h in history], "o-", label="train")\n        axes[1].plot(epochs_axis, [h["val_acc"] for h in history], "o-", label="validation")\n        axes[1].set_title("Accuracy")\n        for ax in axes:\n            ax.set_xlabel("epoch")\n            ax.legend()\n        fig.suptitle(f"{dataset_dict[\'title\']}: {label} ({prep_cfg.tag()})")\n        fig.tight_layout()\n        Path(loss_plot_path).parent.mkdir(parents=True, exist_ok=True)\n        fig.savefig(loss_plot_path, dpi=120)\n        if show:\n            plt.show()\n        plt.close(fig)\n    return metrics\n\n\ndef run_dl_pipeline(\n    dataset_dict: dict,\n    dl_model: str,\n    dl_embedding: str,\n    prep_cfg: PreprocessConfig | None = None,\n    loss_plot_path: Path | None = None,\n    show: bool = False,\n) -> dict | None:\n    """Dispatch the bonus deep-learning path (Word2Vec + BiLSTM)."""\n    if dl_model not in DL_MODELS:\n        raise ValueError(f"Unknown dl_model \'{dl_model}\'")\n    if dl_embedding not in DL_EMBEDDINGS:\n        raise ValueError(f"Unknown dl_embedding \'{dl_embedding}\'")\n    prep_cfg = prep_cfg or dataset_dict.get("preprocess") or PreprocessConfig()\n    return run_bilstm_pipeline(\n        dataset_dict, prep_cfg, dl_embedding, loss_plot_path=loss_plot_path, show=show\n    )\n', 'eda.py': '"""\nExploratory data analysis required by the assignment: word-count statistics,\ntext-length distribution, class balance and word frequencies (with and without\nstop words). Figures go to reports/figures/<dataset>/.\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom collections import Counter\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nfrom modules.config import FIG_DIR\nfrom modules.data_loader import STOP_WORDS, normalize_text, word_counts\n\n_WORD = re.compile(r"[a-z]+(?:\'[a-z]+)?")\n# Function words that carry polarity or direction; highlighted in the word-frequency plot.\nSYNTACTIC_OPERATORS = {\n    "not", "no", "never", "nor", "nothing", "none", "cannot", "without",\n    "still", "into", "from", "but", "too", "very", "can", "how", "why", "again",\n}\n\n\ndef _words(texts):\n    for text in texts:\n        yield from _WORD.findall(normalize_text(text))\n\n\ndef _finish(fig, path: Path, show: bool):\n    fig.tight_layout()\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(path, dpi=120)\n    if show:\n        plt.show()\n    plt.close(fig)\n\n\ndef run_eda(dataset_dict: dict, dataset_name: str | None = None, fig_dir: Path | None = None,\n            show: bool = False, top_k: int = 15) -> pd.DataFrame:\n    """Print EDA statistics, save three figures, and return a one-row summary DataFrame."""\n    name = dataset_name or dataset_dict["name"]\n    out = Path(fig_dir) if fig_dir is not None else FIG_DIR / name\n    title = dataset_dict["title"]\n    train_raw, test_raw = dataset_dict["train_raw"], dataset_dict["test_raw"]\n    labels, names = dataset_dict["train_label"], dataset_dict["label_names"]\n\n    train_len = word_counts([normalize_text(t) for t in train_raw])\n    test_len = word_counts([normalize_text(t) for t in test_raw])\n    counts = np.bincount(labels, minlength=len(names))\n    all_counts = Counter(_words(train_raw))\n    content_counts = Counter({w: c for w, c in all_counts.items() if w not in STOP_WORDS})\n    total_tokens = sum(all_counts.values())\n    stop_tokens = total_tokens - sum(content_counts.values())\n    operator_tokens = sum(all_counts[w] for w in SYNTACTIC_OPERATORS & STOP_WORDS)\n    docs_with_negation = np.mean([\n        bool({"not", "no", "never", "nothing", "cannot"} & set(_WORD.findall(normalize_text(t))))\n        or "n\'t" in normalize_text(t)\n        for t in train_raw\n    ])\n\n    summary = {\n        "dataset": title,\n        "n_train": len(train_raw),\n        "n_test": len(test_raw),\n        "n_classes": len(names),\n        "min_class_count": int(counts.min()),\n        "max_class_count": int(counts.max()),\n        "balance_ratio": round(float(counts.min() / counts.max()), 3),\n        "words_mean": round(float(train_len.mean()), 2),\n        "words_std": round(float(train_len.std()), 2),\n        "words_min": int(train_len.min()),\n        "words_median": round(float(np.median(train_len)), 1),\n        "words_p95": round(float(np.percentile(train_len, 95)), 1),\n        "words_p99": round(float(np.percentile(train_len, 99)), 1),\n        "words_max": int(train_len.max()),\n        "test_words_mean": round(float(test_len.mean()), 2),\n        "vocab_with_stopwords": len(all_counts),\n        "vocab_without_stopwords": len(content_counts),\n        "stopword_token_share": round(stop_tokens / max(total_tokens, 1), 3),\n        "operator_token_share": round(operator_tokens / max(total_tokens, 1), 4),\n        "docs_with_negation": round(float(docs_with_negation), 3),\n    }\n\n    print(f"--- EDA: {title} ---")\n    print(f"train/test: {summary[\'n_train\']}/{summary[\'n_test\']}, classes: {summary[\'n_classes\']}, "\n          f"class counts min/max: {summary[\'min_class_count\']}/{summary[\'max_class_count\']} "\n          f"(ratio {summary[\'balance_ratio\']})")\n    print(f"words/doc: mean={summary[\'words_mean\']}, std={summary[\'words_std\']}, "\n          f"median={summary[\'words_median\']:.0f}, p95={summary[\'words_p95\']:.0f}, "\n          f"p99={summary[\'words_p99\']:.0f}, max={summary[\'words_max\']}")\n    print(f"vocabulary: {summary[\'vocab_with_stopwords\']} with stop words, "\n          f"{summary[\'vocab_without_stopwords\']} without; stop words are "\n          f"{summary[\'stopword_token_share\']:.1%} of tokens; "\n          f"{summary[\'docs_with_negation\']:.1%} of documents contain a negation")\n\n    # 1. Class balance\n    many = len(names) > 20\n    order = np.argsort(-counts) if many else np.arange(len(names))\n    fig, ax = plt.subplots(figsize=(12 if many else 7, 4))\n    ax.bar(range(len(names)), counts[order], color="#3d5a80")\n    ax.axhline(counts.mean(), color="#ee6c4d", ls="--", lw=1, label=f"mean = {counts.mean():.0f}")\n    ax.set_title(f"{title}: training documents per class")\n    ax.set_ylabel("documents")\n    if many:\n        ax.set_xlabel(f"{len(names)} intents, sorted by frequency")\n        ax.set_xticks([])\n    else:\n        ax.set_xticks(range(len(names)))\n        ax.set_xticklabels([names[i] for i in order], rotation=0)\n    ax.legend()\n    _finish(fig, out / "class_balance.png", show)\n\n    # 2. Length distribution with median and p95 padding cut-off\n    cap = int(np.percentile(train_len, 99))\n    fig, ax = plt.subplots(figsize=(7, 4))\n    ax.hist(train_len.clip(max=cap), bins=min(40, max(cap, 5)), color="#98c1d9", edgecolor="white")\n    med, p95 = np.median(train_len), np.percentile(train_len, 95)\n    ax.axvline(med, color="#293241", ls="--", label=f"median = {med:.0f}")\n    ax.axvline(p95, color="#ee6c4d", ls="--", label=f"95th pct (padding cut-off) = {p95:.0f}")\n    ax.set_title(f"{title}: words per training document")\n    ax.set_xlabel("word count (clipped at the 99th percentile)")\n    ax.set_ylabel("documents")\n    ax.legend()\n    _finish(fig, out / "length_hist.png", show)\n\n    # 3. Top words with vs without stop words\n    fig, axes = plt.subplots(1, 2, figsize=(11, 5))\n    for ax, counter, label in [\n        (axes[0], all_counts, "with stop words"),\n        (axes[1], content_counts, "stop words removed"),\n    ]:\n        top = counter.most_common(top_k)\n        words = [w for w, _ in top][::-1]\n        freqs = [c for _, c in top][::-1]\n        colors = ["#ee6c4d" if w in SYNTACTIC_OPERATORS else "#3d5a80" for w in words]\n        ax.barh(words, freqs, color=colors)\n        ax.set_title(f"Top {top_k} words ({label})")\n        ax.set_xlabel("frequency in training set")\n    fig.suptitle(f"{title}: word frequencies (orange = negation / direction / question words)")\n    _finish(fig, out / "top_words_comparison.png", show)\n\n    print(f"[eda] figures saved to {out}")\n    return pd.DataFrame([summary])\n', 'evaluation.py': '"""\nExperiment orchestration, result tables and figures for Banking77.\n\nResults are cached in-process by (dataset, preprocessing tag, method) so ablation\nand main comparison always report the same number for a shared configuration.\n\nOutputs\n-------\nreports/comparison_banking77.csv\nreports/ablation_banking77.csv / ablation_delta_banking77.csv\nreports/errors_banking77.csv / negation_banking77.csv\nreports/eda_summary.csv, master_comparison.csv, master_f1_pivot.csv, master_ablation.csv\nreports/figures/banking77/*.png\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nfrom modules.classifiers import train_and_evaluate_classifier\nfrom modules.config import (\n    DATASET_ORDER,\n    DATASET_REGISTRY,\n    DL_EMBEDDINGS,\n    DL_MODELS,\n    FIG_DIR,\n    REPORT_DIR,\n    ExperimentConfig,\n    PreprocessConfig,\n)\nfrom modules.data_loader import load_and_preprocess_dataset, normalize_text\nfrom modules.deep_learning import run_dl_pipeline\nfrom modules.eda import run_eda\nfrom modules.feature_extraction import extract_and_save_features\n\nMAIN_PREP = PreprocessConfig(\n    stop_word_removal=False, tokenization=True, padding=True, negation_marking=False\n)\n\nFLAG_SWITCHES = {\n    "stop_word_removal": (\n        MAIN_PREP,\n        PreprocessConfig(True, True, True, False),\n    ),\n    "tokenization": (\n        PreprocessConfig(False, False, True, False),\n        MAIN_PREP,\n    ),\n    "padding": (\n        PreprocessConfig(False, True, False, False),\n        MAIN_PREP,\n    ),\n    "negation_marking": (\n        MAIN_PREP,\n        PreprocessConfig(False, True, True, True),\n    ),\n}\nABLATION_MODELS = {\n    "TF-IDF + LogReg": ("tfidf", "logreg"),\n    "DistilBERT + LogReg": ("distilbert", "logreg"),\n    "Word2Vec + BiLSTM": ("bilstm", "word2vec"),\n}\n\nEMBEDDINGS = ["tfidf", "word2vec", "distilbert"]\nEMBEDDING_NAMES = {\n    "bow": "BoW",\n    "tfidf": "TF-IDF",\n    "word2vec": "Word2Vec",\n    "distilbert": "DistilBERT",\n}\nCLASSICAL = [("naive_bayes", False), ("logreg", False), ("svm", False)]\nMAIN_GRID = {\n    "tfidf": CLASSICAL,\n    "word2vec": CLASSICAL,\n    "distilbert": CLASSICAL,\n}\nFAMILY = {\n    "bow": "Traditional (sparse lexical)",\n    "tfidf": "Traditional (sparse lexical)",\n    "word2vec": "Static word embeddings",\n    "distilbert": "Modern (frozen Transformer)",\n    "bilstm": "Deep learning (end-to-end)",\n}\nFAMILY_COLORS = {\n    "Traditional (sparse lexical)": "#3d5a80",\n    "Static word embeddings": "#8ab17d",\n    "Modern (frozen Transformer)": "#ee6c4d",\n    "Deep learning (end-to-end)": "#2a9d8f",\n}\nCLASSIFIER_COLORS = {\n    "naive_bayes": "#e76f51",\n    "logreg": "#264653",\n    "svm": "#e9c46a",\n    "bilstm": "#2a9d8f",\n}\nEMBEDDING_MARKERS = {\n    "tfidf": "D",\n    "word2vec": "v",\n    "distilbert": "^",\n    "bilstm": "P",\n}\nCSV_COLUMNS = [\n    "dataset", "method", "family", "embedding", "classifier", "input", "input_dim", "preprocess",\n    "accuracy", "precision", "recall", "f1", "brier_score", "ece",\n    "extract_time_sec", "train_time_sec", "infer_time_sec", "notes",\n]\nNEGATION = re.compile(r"\\b(not|no|never|nothing|nobody|none|cannot|nor|without)\\b|n\'t\\b")\n\n_DATA: dict = {}\n_FEATURES: dict = {}\n_RESULTS: dict = {}\n\n\ndef get_dataset(name: str, prep: PreprocessConfig = MAIN_PREP) -> dict:\n    key = (name, prep.tag())\n    if key not in _DATA:\n        _DATA[key] = load_and_preprocess_dataset(name, prep)\n    return _DATA[key]\n\n\ndef get_features(name: str, embedding: str, prep: PreprocessConfig = MAIN_PREP) -> dict:\n    key = (name, embedding, prep.tag())\n    if key not in _FEATURES:\n        _FEATURES[key] = extract_and_save_features(\n            get_dataset(name, prep), ExperimentConfig(name, prep, embedding)\n        )\n    return _FEATURES[key]\n\n\ndef method_label(embedding: str, classifier: str, projection: bool) -> str:\n    sparse_input = embedding in ("bow", "tfidf")\n    emb = EMBEDDING_NAMES[embedding]\n    if projection:\n        emb += " (SVD-30)" if sparse_input else " (PCA-30)"\n    clf = {\n        "naive_bayes": "MultinomialNB" if sparse_input and not projection else "GaussianNB",\n        "logreg": "LogReg",\n        "svm": "LinearSVC",\n    }[classifier]\n    return f"{emb} + {clf}"\n\n\ndef _result_key(name, prep, method_key):\n    return (name, prep.tag(), method_key)\n\n\ndef evaluate(\n    name: str,\n    embedding: str,\n    classifier: str,\n    projection: bool = False,\n    prep: PreprocessConfig = MAIN_PREP,\n) -> dict:\n    key = _result_key(name, prep, f"{embedding}|{classifier}|{int(projection)}")\n    if key not in _RESULTS:\n        data = get_dataset(name, prep)\n        feats = get_features(name, embedding, prep)\n        m = train_and_evaluate_classifier(\n            feats,\n            data["train_label"],\n            data["test_label"],\n            classifier,\n            method_label(embedding, classifier, projection),\n            use_projection=projection,\n            n_classes=data["n_classes"],\n        )\n        m.update({\n            "embedding": EMBEDDING_NAMES[embedding],\n            "embedding_key": embedding,\n            "family": FAMILY[embedding],\n            "preprocess": prep.tag(),\n            "extract_time_sec": float(feats["meta"].get("extract_time_sec", 0.0)),\n            "notes": "",\n        })\n        _RESULTS[key] = m\n    return _RESULTS[key]\n\n\ndef run_dl_cell(\n    name: str,\n    dl_model: str,\n    dl_embedding: str,\n    prep: PreprocessConfig = MAIN_PREP,\n    show: bool = False,\n) -> dict | None:\n    method_key = f"dl|{dl_model}|{dl_embedding}"\n    key = _result_key(name, prep, method_key)\n    if key not in _RESULTS:\n        data = get_dataset(name, prep)\n        plot = None\n        if dl_model == "bilstm" and dl_embedding == "word2vec" and prep.tag() == MAIN_PREP.tag():\n            plot = FIG_DIR / name / "bilstm_loss.png"\n        m = run_dl_pipeline(data, dl_model, dl_embedding, prep, loss_plot_path=plot, show=show)\n        if m is None:\n            return None\n        emb_name = EMBEDDING_NAMES.get(dl_embedding, dl_embedding)\n        m.update({\n            "embedding": emb_name,\n            "embedding_key": dl_model,\n            "family": FAMILY[dl_model],\n            "preprocess": prep.tag(),\n            "extract_time_sec": 0.0,\n            "notes": (\n                f"{dl_model} x {dl_embedding}"\n                + (f", {m[\'epochs_run\']} epochs" if "epochs_run" in m else "")\n            ),\n        })\n        _RESULTS[key] = m\n    return _RESULTS[key]\n\n\ndef _row(name: str, m: dict) -> dict:\n    row = {col: m.get(col) for col in CSV_COLUMNS}\n    row["dataset"] = DATASET_REGISTRY[name]["title"]\n    row["notes"] = m.get("notes", "")\n    return row\n\n\ndef _main_keys():\n    for emb in EMBEDDINGS:\n        for clf, proj in MAIN_GRID[emb]:\n            yield emb, clf, proj, f"{emb}|{clf}|{int(proj)}"\n    for dl_emb in DL_EMBEDDINGS:\n        for dl_model in DL_MODELS:\n            yield dl_emb, dl_model, False, f"dl|{dl_model}|{dl_emb}"\n\n\ndef collected_results(name: str) -> list[dict]:\n    out = []\n    for *_, method_key in _main_keys():\n        m = _RESULTS.get(_result_key(name, MAIN_PREP, method_key))\n        if m is not None:\n            out.append(m)\n    return out\n\n\ndef run_all_eda(datasets=DATASET_ORDER, show: bool = False) -> pd.DataFrame:\n    frames = [run_eda(get_dataset(name, MAIN_PREP), name, show=show) for name in datasets]\n    table = pd.concat(frames, ignore_index=True)\n    REPORT_DIR.mkdir(parents=True, exist_ok=True)\n    table.to_csv(REPORT_DIR / "eda_summary.csv", index=False)\n    return table\n\n\ndef run_feature_extraction(name: str, embeddings=EMBEDDINGS) -> pd.DataFrame:\n    rows = []\n    for emb in embeddings:\n        f = get_features(name, emb, MAIN_PREP)\n        rows.append({\n            "dataset": DATASET_REGISTRY[name]["title"],\n            "embedding": EMBEDDING_NAMES[emb],\n            "full_dim": f["dim"],\n            "sparse": f["is_sparse"],\n            "projection": f["meta"]["projection"],\n            "variance_kept_30d": f["meta"]["explained_variance_30d"],\n            "max_length": f["meta"].get("max_length"),\n            "extract_time_sec": round(float(f["meta"]["extract_time_sec"]), 2),\n            "folder": str(Path(f["cache_dir"]).relative_to(REPORT_DIR.parent)),\n        })\n    return pd.DataFrame(rows)\n\n\ndef run_main_comparison(name: str, embeddings=EMBEDDINGS) -> pd.DataFrame:\n    print(f"\\n== {DATASET_REGISTRY[name][\'title\']}: embedding x classifier matrix ({MAIN_PREP.tag()}) ==")\n    for emb in embeddings:\n        for clf, proj in MAIN_GRID[emb]:\n            evaluate(name, emb, clf, proj)\n    return save_comparison(name)\n\n\ndef run_deep_learning(name: str, run_bilstm_model: bool = True, show: bool = False) -> pd.DataFrame:\n    print(f"\\n== {DATASET_REGISTRY[name][\'title\']}: bonus deep learning (Word2Vec + BiLSTM) ==")\n    rows = []\n    if run_bilstm_model:\n        for dl_emb in DL_EMBEDDINGS:\n            for dl_model in DL_MODELS:\n                m = run_dl_cell(name, dl_model, dl_emb, MAIN_PREP, show=show)\n                if m is not None:\n                    rows.append(_row(name, m))\n    save_comparison(name)\n    return pd.DataFrame(rows, columns=CSV_COLUMNS)\n\n\ndef save_comparison(name: str) -> pd.DataFrame:\n    table = pd.DataFrame([_row(name, m) for m in collected_results(name)], columns=CSV_COLUMNS)\n    REPORT_DIR.mkdir(parents=True, exist_ok=True)\n    table.round(6).to_csv(REPORT_DIR / f"comparison_{name}.csv", index=False)\n    return table\n\n\ndef run_preprocessing_ablation(name: str, include_bilstm: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """One-flag-at-a-time ablation around MAIN_PREP for the three ablation models."""\n    print(f"\\n== {DATASET_REGISTRY[name][\'title\']}: boolean preprocessing ablation ==")\n    runs, deltas = {}, []\n    for model_name, (emb, clf) in ABLATION_MODELS.items():\n        if emb == "bilstm" and not include_bilstm:\n            continue\n        for flag, (cfg_off, cfg_on) in FLAG_SWITCHES.items():\n            if flag == "padding" and emb in ("bow", "tfidf"):\n                deltas.append({\n                    "dataset": DATASET_REGISTRY[name]["title"],\n                    "model": model_name,\n                    "flag": flag,\n                    "config_false": cfg_off.tag(),\n                    "config_true": cfg_on.tag(),\n                    "f1_false": np.nan,\n                    "f1_true": np.nan,\n                    "delta_f1": np.nan,\n                    "acc_false": np.nan,\n                    "acc_true": np.nan,\n                    "note": "not applicable: bag-of-words has no sequence length",\n                })\n                continue\n            results = []\n            for cfg in (cfg_off, cfg_on):\n                if emb == "bilstm":\n                    m = run_dl_cell(name, "bilstm", clf or "word2vec", cfg)\n                else:\n                    m = evaluate(name, emb, clf, False, cfg)\n                if m is None:\n                    continue\n                runs[(model_name, cfg.tag())] = {\n                    "dataset": DATASET_REGISTRY[name]["title"],\n                    "model": model_name,\n                    "preprocess": cfg.tag(),\n                    "stop_word_removal": cfg.stop_word_removal,\n                    "tokenization": cfg.tokenization,\n                    "padding": cfg.padding,\n                    "negation_marking": cfg.negation_marking,\n                    "accuracy": m["accuracy"],\n                    "f1": m["f1"],\n                    "brier_score": m["brier_score"],\n                    "notes": m.get("notes", ""),\n                }\n                results.append(m)\n            if len(results) == 2:\n                deltas.append({\n                    "dataset": DATASET_REGISTRY[name]["title"],\n                    "model": model_name,\n                    "flag": flag,\n                    "config_false": cfg_off.tag(),\n                    "config_true": cfg_on.tag(),\n                    "f1_false": results[0]["f1"],\n                    "f1_true": results[1]["f1"],\n                    "delta_f1": results[1]["f1"] - results[0]["f1"],\n                    "acc_false": results[0]["accuracy"],\n                    "acc_true": results[1]["accuracy"],\n                    "note": "",\n                })\n    runs_df = pd.DataFrame(list(runs.values()))\n    delta_df = pd.DataFrame(deltas)\n    runs_df.round(6).to_csv(REPORT_DIR / f"ablation_{name}.csv", index=False)\n    delta_df.round(6).to_csv(REPORT_DIR / f"ablation_delta_{name}.csv", index=False)\n    return runs_df, delta_df\n\n\ndef best_result(name: str) -> dict:\n    return max(collected_results(name), key=lambda m: m["f1"])\n\n\ndef error_analysis(name: str, top: int = 10) -> pd.DataFrame:\n    data = get_dataset(name, MAIN_PREP)\n    best = best_result(name)\n    y, pred, names = data["test_label"], best["predictions"], data["label_names"]\n    wrong = np.where(y != pred)[0]\n    pairs = pd.Series(list(zip(y[wrong], pred[wrong]))).value_counts().head(top)\n    rows = []\n    for (t, p), count in pairs.items():\n        examples = [data["test_raw"][i] for i in wrong if y[i] == t and pred[i] == p][:2]\n        rows.append({\n            "dataset": data["title"],\n            "model": best["method"],\n            "true": names[t],\n            "predicted": names[p],\n            "count": int(count),\n            "share_of_true_class": round(count / max((y == t).sum(), 1), 3),\n            "example_1": normalize_text(examples[0])[:200] if examples else "",\n            "example_2": normalize_text(examples[1])[:200] if len(examples) > 1 else "",\n        })\n    table = pd.DataFrame(rows)\n    table.to_csv(REPORT_DIR / f"errors_{name}.csv", index=False)\n    return table\n\n\ndef negation_analysis(name: str) -> pd.DataFrame:\n    data = get_dataset(name, MAIN_PREP)\n    has_neg = np.array([bool(NEGATION.search(normalize_text(t))) for t in data["test_raw"]])\n    rows = []\n    for m in collected_results(name):\n        correct = m["predictions"] == data["test_label"]\n        rows.append({\n            "dataset": data["title"],\n            "method": m["method"],\n            "n_negation": int(has_neg.sum()),\n            "n_other": int((~has_neg).sum()),\n            "acc_negation": float(correct[has_neg].mean()) if has_neg.any() else np.nan,\n            "acc_other": float(correct[~has_neg].mean()) if (~has_neg).any() else np.nan,\n        })\n    table = pd.DataFrame(rows)\n    table["gap"] = table["acc_negation"] - table["acc_other"]\n    table.round(6).to_csv(REPORT_DIR / f"negation_{name}.csv", index=False)\n    return table\n\n\ndef _finish(fig, path: Path, show: bool):\n    fig.tight_layout()\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(path, dpi=120)\n    if show:\n        plt.show()\n    plt.close(fig)\n\n\ndef plot_accuracy_f1(name: str, results_df: pd.DataFrame, show: bool = False):\n    df = results_df.sort_values("f1")\n    fig, ax = plt.subplots(figsize=(9, 0.36 * len(df) + 1.4))\n    ypos = np.arange(len(df))\n    colors = [FAMILY_COLORS.get(f, "#888888") for f in df["family"]]\n    ax.barh(ypos + 0.2, df["accuracy"] * 100, height=0.4, color=colors, alpha=0.45, label="Accuracy")\n    ax.barh(ypos - 0.2, df["f1"] * 100, height=0.4, color=colors, label="Macro-F1")\n    for y, f1 in zip(ypos, df["f1"]):\n        ax.text(f1 * 100 + 0.3, y - 0.2, f"{f1 * 100:.1f}", va="center", fontsize=7)\n    ax.set_yticks(ypos)\n    ax.set_yticklabels(df["method"], fontsize=8)\n    lo = max(0, np.floor(min(df["f1"].min(), df["accuracy"].min()) * 100 / 5) * 5 - 5)\n    ax.set_xlim(lo, 100)\n    ax.set_xlabel("score (%)  [solid = Macro-F1, light = Accuracy]")\n    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FAMILY_COLORS.values()]\n    ax.legend(handles, FAMILY_COLORS.keys(), loc="lower right", fontsize=8)\n    ax.set_title(f"{DATASET_REGISTRY[name][\'title\']}: test Accuracy and Macro-F1 ({MAIN_PREP.tag()})")\n    _finish(fig, FIG_DIR / name / "accuracy_f1_bars.png", show)\n\n\ndef plot_calibration_scatter(name: str, show: bool = False):\n    ms = collected_results(name)\n    fig, ax = plt.subplots(figsize=(8, 5.5))\n    for m in ms:\n        ax.scatter(\n            m["brier_score"],\n            m["f1"] * 100,\n            s=90,\n            color=CLASSIFIER_COLORS.get(m["classifier"], "#888"),\n            marker=EMBEDDING_MARKERS.get(m["embedding_key"], "o"),\n            edgecolor="black",\n            linewidth=0.5,\n        )\n        ax.annotate(\n            m["method"],\n            (m["brier_score"], m["f1"] * 100),\n            fontsize=6,\n            xytext=(4, 2),\n            textcoords="offset points",\n        )\n    clf_handles = [\n        plt.Line2D([], [], marker="o", ls="", color=c, label=k) for k, c in CLASSIFIER_COLORS.items()\n    ]\n    emb_handles = [\n        plt.Line2D([], [], marker=mk, ls="", color="grey", label=k)\n        for k, mk in EMBEDDING_MARKERS.items()\n    ]\n    first = ax.legend(handles=clf_handles, title="classifier", fontsize=7, title_fontsize=8, loc="lower left")\n    ax.add_artist(first)\n    ax.legend(handles=emb_handles, title="representation", fontsize=7, title_fontsize=8, loc="upper right")\n    ax.set_xlabel("multiclass Brier score (lower = better calibrated)")\n    ax.set_ylabel("Macro-F1 (%)")\n    ax.set_title(f"{DATASET_REGISTRY[name][\'title\']}: accuracy vs probability calibration")\n    _finish(fig, FIG_DIR / name / "brier_calibration_vs_f1.png", show)\n\n\ndef plot_reliability(name: str, show: bool = False, n_bins: int = 10):\n    data = get_dataset(name, MAIN_PREP)\n    wanted = [\n        "tfidf|naive_bayes|0",\n        "tfidf|logreg|0",\n        "tfidf|svm|0",\n        "distilbert|logreg|0",\n        "distilbert|svm|0",\n        "dl|bilstm|word2vec",\n    ]\n    fig, ax = plt.subplots(figsize=(6, 5.5))\n    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")\n    edges = np.linspace(0, 1, n_bins + 1)\n    for key in wanted:\n        m = _RESULTS.get(_result_key(name, MAIN_PREP, key))\n        if m is None:\n            continue\n        conf = m["probabilities"].max(1)\n        correct = m["predictions"] == data["test_label"]\n        xs, ys = [], []\n        for lo, hi in zip(edges[:-1], edges[1:]):\n            sel = (conf > lo) & (conf <= hi)\n            if sel.sum() >= 5:\n                xs.append(conf[sel].mean())\n                ys.append(correct[sel].mean())\n        ax.plot(\n            xs,\n            ys,\n            "o-",\n            color=CLASSIFIER_COLORS.get(m["classifier"], "#888"),\n            ms=4,\n            label=f"{m[\'method\']} (ECE {m[\'ece\']:.3f})",\n        )\n    ax.set_xlabel("predicted confidence (top class)")\n    ax.set_ylabel("observed accuracy")\n    ax.set_xlim(0, 1)\n    ax.set_ylim(0, 1.02)\n    ax.legend(fontsize=7, loc="upper left")\n    ax.set_title(f"{DATASET_REGISTRY[name][\'title\']}: reliability diagram")\n    _finish(fig, FIG_DIR / name / "reliability_diagram.png", show)\n\n\ndef plot_ablation_heatmap(name: str, delta_df: pd.DataFrame, show: bool = False):\n    pivot = delta_df.pivot(index="model", columns="flag", values="delta_f1")\n    pivot = pivot.reindex(\n        index=[m for m in ABLATION_MODELS if m in pivot.index],\n        columns=list(FLAG_SWITCHES),\n    ) * 100\n    vmax = max(1.0, float(np.nanmax(np.abs(pivot.values))))\n    fig, ax = plt.subplots(figsize=(8.5, 3.4))\n    image = ax.imshow(pivot.values, cmap="RdBu", vmin=-vmax, vmax=vmax, aspect="auto")\n    ax.set_xticks(range(pivot.shape[1]))\n    ax.set_xticklabels([f"{c}\\n(True - False)" for c in pivot.columns], fontsize=8)\n    ax.set_yticks(range(pivot.shape[0]))\n    ax.set_yticklabels(pivot.index, fontsize=8)\n    for i in range(pivot.shape[0]):\n        for j in range(pivot.shape[1]):\n            v = pivot.values[i, j]\n            ax.text(\n                j,\n                i,\n                "n/a" if np.isnan(v) else f"{v:+.2f}",\n                ha="center",\n                va="center",\n                fontsize=9,\n                color="black" if np.isnan(v) or abs(v) < 0.6 * vmax else "white",\n            )\n    fig.colorbar(image, ax=ax, label="delta Macro-F1 (pp)")\n    ax.set_title(f"{DATASET_REGISTRY[name][\'title\']}: effect of switching each flag on")\n    _finish(fig, FIG_DIR / name / "preprocessing_ablation_heatmap.png", show)\n\n\ndef plot_confusion_best(name: str, show: bool = False):\n    from sklearn.metrics import confusion_matrix\n\n    data = get_dataset(name, MAIN_PREP)\n    best = best_result(name)\n    names = data["label_names"]\n    mat = confusion_matrix(data["test_label"], best["predictions"], labels=np.arange(len(names)))\n    shown = mat / np.maximum(mat.sum(axis=1, keepdims=True), 1)\n    many = len(names) > 10\n    fig, ax = plt.subplots(figsize=(9, 8) if many else (5.5, 4.8))\n    image = ax.imshow(shown, cmap="Blues", vmin=0, vmax=1)\n    if many:\n        ax.set_xlabel(f"predicted intent (0-{len(names) - 1})")\n        ax.set_ylabel(f"true intent (0-{len(names) - 1})")\n    else:\n        ax.set_xticks(range(len(names)))\n        ax.set_xticklabels(names, rotation=30, ha="right")\n        ax.set_yticks(range(len(names)))\n        ax.set_yticklabels(names)\n        for i in range(len(names)):\n            for j in range(len(names)):\n                ax.text(\n                    j,\n                    i,\n                    f"{shown[i, j]:.2f}",\n                    ha="center",\n                    va="center",\n                    fontsize=8,\n                    color="white" if shown[i, j] > 0.5 else "black",\n                )\n        ax.set_xlabel("predicted")\n        ax.set_ylabel("true")\n    fig.colorbar(image, ax=ax, fraction=0.046, label="row-normalized rate")\n    ax.set_title(f"{data[\'title\']}: {best[\'method\']} (Macro-F1 {best[\'f1\'] * 100:.2f}%)")\n    _finish(fig, FIG_DIR / name / "confusion_best.png", show)\n\n\ndef plot_suite_results(name: str, results_df: pd.DataFrame, delta_df: pd.DataFrame | None, show: bool = False):\n    plot_accuracy_f1(name, results_df, show)\n    plot_calibration_scatter(name, show)\n    plot_reliability(name, show)\n    if delta_df is not None and len(delta_df):\n        plot_ablation_heatmap(name, delta_df, show)\n    plot_confusion_best(name, show)\n\n\ndef run_dataset_benchmark_suite(\n    name: str,\n    run_ablation: bool = True,\n    run_bilstm_model: bool = True,\n    show: bool = False,\n) -> tuple[pd.DataFrame, pd.DataFrame | None]:\n    """Ablation + main matrix + Word2Vec BiLSTM + figures + error analysis."""\n    delta_df = None\n    if run_ablation:\n        _, delta_df = run_preprocessing_ablation(name, include_bilstm=run_bilstm_model)\n    run_main_comparison(name)\n    run_deep_learning(name, run_bilstm_model=run_bilstm_model, show=show)\n    results_df = save_comparison(name)\n    plot_suite_results(name, results_df, delta_df, show)\n    error_analysis(name)\n    negation_analysis(name)\n    return results_df, delta_df\n\n\ndef build_master_table(datasets=DATASET_ORDER, show: bool = False) -> pd.DataFrame:\n    frames = [\n        pd.read_csv(REPORT_DIR / f"comparison_{n}.csv")\n        for n in datasets\n        if (REPORT_DIR / f"comparison_{n}.csv").exists()\n    ]\n    master = pd.concat(frames, ignore_index=True)\n    master.to_csv(REPORT_DIR / "master_comparison.csv", index=False)\n    titles = [DATASET_REGISTRY[n]["title"] for n in datasets]\n    pivot = master.pivot_table(index="method", columns="dataset", values="f1", sort=False)\n    pivot = pivot[[t for t in titles if t in pivot.columns]]\n    pivot.round(6).to_csv(REPORT_DIR / "master_f1_pivot.csv")\n\n    ablations = [\n        pd.read_csv(REPORT_DIR / f"ablation_delta_{n}.csv")\n        for n in datasets\n        if (REPORT_DIR / f"ablation_delta_{n}.csv").exists()\n    ]\n    if ablations:\n        pd.concat(ablations, ignore_index=True).to_csv(REPORT_DIR / "master_ablation.csv", index=False)\n\n    fig, ax = plt.subplots(figsize=(6.5, 0.4 * len(pivot) + 1.5))\n    image = ax.imshow(pivot.values * 100, cmap="viridis", aspect="auto")\n    ax.set_xticks(range(pivot.shape[1]))\n    ax.set_xticklabels(pivot.columns)\n    ax.set_yticks(range(pivot.shape[0]))\n    ax.set_yticklabels(pivot.index, fontsize=8)\n    for i in range(pivot.shape[0]):\n        for j in range(pivot.shape[1]):\n            v = pivot.values[i, j] * 100\n            if not np.isnan(v):\n                ax.text(\n                    j,\n                    i,\n                    f"{v:.1f}",\n                    ha="center",\n                    va="center",\n                    fontsize=8,\n                    color="white" if v < np.nanmean(pivot.values) * 100 else "black",\n                )\n    fig.colorbar(image, ax=ax, label="Macro-F1 (%)")\n    ax.set_title(f"Macro-F1 (default preprocessing {MAIN_PREP.tag()})")\n    _finish(fig, FIG_DIR / "master_f1_heatmap.png", show)\n    return master\n', 'feature_extraction.py': '"""\nFeature extraction with on-disk persistence in both .npy and .h5 (HDF5).\n\nEmbedding types\n---------------\nbow        CountVectorizer (sparse) + 30-d TruncatedSVD (optional plug-in)\ntfidf      sublinear TfidfVectorizer (sparse) + 30-d TruncatedSVD\nword2vec   skip-gram Word2Vec on train texts (300-d mean pool) + PCA-30;\n           also saves word2vec.kv.npz for BiLSTM reuse\ndistilbert frozen distilbert-base-uncased, masked mean pooling (768-d) + PCA-30\n\nFiles written to features/<dataset>/<embedding>_<preprocess tag>/\n-----------------------------------------------------------------\nX_train.npy, X_test.npy              full dense matrix (dense embeddings only)\nX_train_sparse.npz, X_test_sparse.npz full sparse matrix (bow / tfidf only)\nX_train_pca30.npy, X_test_pca30.npy  30-d projection (SVD for sparse, PCA for dense)\ny_train.npy, y_test.npy              integer labels\nfeatures.h5                          every array above in one HDF5 file\nmeta.json                            settings used to build the features\nword2vec.kv.npz                      cached KeyedVectors (word2vec only)\n\nHow the boolean preprocessing switches reach the extractors\n-----------------------------------------------------------\nstop_word_removal / negation_marking  already applied by data_loader.clean_text\ntokenization       sparse: True -> regex words + bigrams, False -> str.split unigrams\n                   neural: True -> WordPiece subwords, False -> whole-word vocabulary\n                   lookup (out-of-vocabulary words become [UNK], no subword split)\n                   word2vec: always whitespace tokens (static word table)\npadding            neural: True -> truncate at the 95th-percentile token length\n                   (capped per dataset) and mask padding in attention and pooling;\n                   False -> truncate at 10 tokens and pool over padding unmasked\n                   sparse / word2vec mean-pool: not applicable\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy import sparse\nfrom sklearn.decomposition import PCA, TruncatedSVD\nfrom sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer\nfrom sklearn.preprocessing import StandardScaler\n\nfrom modules.config import (\n    BERT_BATCH_SIZE,\n    BERT_MODEL,\n    FEATURE_DIR,\n    MAX_FEATURES,\n    PCA_DIM,\n    SEED,\n    ExperimentConfig,\n)\nfrom modules.utils import get_device\n\nSPARSE_TYPES = ("bow", "tfidf")\nDENSE_MODELS = {\n    "distilbert": {"model_id": BERT_MODEL, "pooling": "mean", "normalize": False},\n}\nUNPADDED_MAX_LEN = 10\n\n\ndef cache_dir_for(dataset_name: str, embedding_type: str, tag: str, feature_root: Path = FEATURE_DIR) -> Path:\n    return Path(feature_root) / dataset_name / f"{embedding_type}_{tag}"\n\n\n# ---------------------------------------------------------------------------\n# Persistence\n# ---------------------------------------------------------------------------\ndef _save(folder: Path, arrays: dict, sparse_arrays: dict, meta: dict, save_format: str) -> None:\n    import h5py\n\n    folder.mkdir(parents=True, exist_ok=True)\n    for key, mat in sparse_arrays.items():\n        sparse.save_npz(folder / f"{key}.npz", mat.tocsr())\n    if save_format in ("npy", "both"):\n        for key, arr in arrays.items():\n            np.save(folder / f"{key}.npy", arr)\n    if save_format in ("h5", "both"):\n        with h5py.File(folder / "features.h5", "w") as h5:\n            for key, arr in arrays.items():\n                h5.create_dataset(key, data=arr, compression="gzip", compression_opts=4)\n            for key, mat in sparse_arrays.items():\n                grp = h5.create_group(key)\n                csr = mat.tocsr()\n                grp.create_dataset("data", data=csr.data, compression="gzip", compression_opts=4)\n                grp.create_dataset("indices", data=csr.indices, compression="gzip", compression_opts=4)\n                grp.create_dataset("indptr", data=csr.indptr)\n                grp.attrs["shape"] = csr.shape\n            h5.attrs["meta"] = json.dumps(meta)\n    (folder / "meta.json").write_text(json.dumps(meta, indent=2))\n\n\ndef load_h5_features(path: Path) -> dict:\n    """Read a features.h5 file back into numpy / scipy objects (used to verify the .h5 copy)."""\n    import h5py\n\n    out = {}\n    with h5py.File(path, "r") as h5:\n        for key in h5.keys():\n            node = h5[key]\n            if isinstance(node, h5py.Group):\n                out[key] = sparse.csr_matrix(\n                    (node["data"][:], node["indices"][:], node["indptr"][:]), shape=tuple(node.attrs["shape"])\n                )\n            else:\n                out[key] = node[:]\n        out["meta"] = json.loads(h5.attrs["meta"])\n    return out\n\n\ndef _load_cached(folder: Path, is_sparse: bool, y_train, y_test):\n    needed = ["X_train_pca30.npy", "X_test_pca30.npy", "y_train.npy", "y_test.npy", "meta.json", "features.h5"]\n    needed += ["X_train_sparse.npz", "X_test_sparse.npz"] if is_sparse else ["X_train.npy", "X_test.npy"]\n    if not all((folder / f).exists() for f in needed):\n        return None\n    if not (np.array_equal(np.load(folder / "y_train.npy"), y_train)\n            and np.array_equal(np.load(folder / "y_test.npy"), y_test)):\n        return None\n    if is_sparse:\n        x_train, x_test = sparse.load_npz(folder / "X_train_sparse.npz"), sparse.load_npz(folder / "X_test_sparse.npz")\n    else:\n        x_train, x_test = np.load(folder / "X_train.npy"), np.load(folder / "X_test.npy")\n    return {\n        "X_train": x_train,\n        "X_test": x_test,\n        "X_train_pca30": np.load(folder / "X_train_pca30.npy"),\n        "X_test_pca30": np.load(folder / "X_test_pca30.npy"),\n        "meta": json.loads((folder / "meta.json").read_text()),\n    }\n\n\n# ---------------------------------------------------------------------------\n# Sparse lexical features\n# ---------------------------------------------------------------------------\ndef build_vectorizer(embedding_type: str, tokenization: bool):\n    common = dict(max_features=MAX_FEATURES, min_df=2, lowercase=True, dtype=np.float32)\n    if tokenization:\n        common.update(ngram_range=(1, 2))\n    else:\n        common.update(ngram_range=(1, 1), tokenizer=str.split, token_pattern=None)\n    if embedding_type == "bow":\n        return CountVectorizer(**common)\n    return TfidfVectorizer(sublinear_tf=True, **common)\n\n\ndef _sparse_features(data: dict, cfg: ExperimentConfig):\n    vec = build_vectorizer(cfg.embedding_type, cfg.preprocess.tokenization)\n    x_train = vec.fit_transform(data["train_text"]).astype(np.float32)\n    x_test = vec.transform(data["test_text"]).astype(np.float32)\n    svd = TruncatedSVD(n_components=PCA_DIM, random_state=SEED)\n    z_train = svd.fit_transform(x_train).astype(np.float32)\n    z_test = svd.transform(x_test).astype(np.float32)\n    meta = {\n        "vocabulary_size": len(vec.vocabulary_),\n        "ngram_range": list(vec.ngram_range),\n        "projection": f"TruncatedSVD({PCA_DIM})",\n        "explained_variance_30d": round(float(svd.explained_variance_ratio_.sum()), 4),\n    }\n    return x_train, x_test, z_train, z_test, meta\n\n\n# ---------------------------------------------------------------------------\n# Dense Transformer features\n# ---------------------------------------------------------------------------\ndef _word_level_batch(texts, tokenizer, max_length):\n    """Whole-word vocabulary lookup without subword decomposition (tokenization=False)."""\n    import torch\n\n    vocab = tokenizer.get_vocab()\n    unk, cls, sep, pad = (tokenizer.unk_token_id, tokenizer.cls_token_id,\n                          tokenizer.sep_token_id, tokenizer.pad_token_id)\n    rows = [[cls] + [vocab.get(w, unk) for w in t.split()][: max_length - 2] + [sep] for t in texts]\n    width = max(len(r) for r in rows)\n    ids = torch.full((len(rows), width), pad, dtype=torch.long)\n    mask = torch.zeros((len(rows), width), dtype=torch.long)\n    for i, r in enumerate(rows):\n        ids[i, : len(r)] = torch.tensor(r)\n        mask[i, : len(r)] = 1\n    return {"input_ids": ids, "attention_mask": mask}\n\n\ndef token_lengths(texts, tokenizer, tokenization: bool) -> np.ndarray:\n    if tokenization:\n        enc = tokenizer(list(texts), add_special_tokens=True, truncation=False)["input_ids"]\n        return np.array([len(e) for e in enc])\n    return np.array([len(t.split()) + 2 for t in texts])\n\n\ndef resolve_max_length(train_text, tokenizer, prep, cap: int) -> int:\n    if not prep.padding:\n        return UNPADDED_MAX_LEN\n    p95 = int(np.ceil(np.percentile(token_lengths(train_text, tokenizer, prep.tokenization), 95)))\n    return int(min(cap, max(24, p95)))\n\n\ndef _encode(texts, tokenizer, model, device, prep, max_length, pooling, normalize):\n    import torch\n\n    order = np.argsort([len(t) for t in texts])\n    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float32)\n    model.eval()\n    with torch.no_grad():\n        for start in range(0, len(texts), BERT_BATCH_SIZE):\n            idx = order[start : start + BERT_BATCH_SIZE]\n            batch = [texts[i] for i in idx]\n            if prep.tokenization:\n                enc = tokenizer(batch, padding="longest", truncation=True, max_length=max_length,\n                                return_tensors="pt")\n                enc = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}\n            else:\n                enc = _word_level_batch(batch, tokenizer, max_length)\n            mask = enc["attention_mask"].to(device)\n            ids = enc["input_ids"].to(device)\n            if prep.padding:\n                hidden = model(input_ids=ids, attention_mask=mask).last_hidden_state\n            else:\n                hidden = model(input_ids=ids, attention_mask=torch.ones_like(mask)).last_hidden_state\n            if pooling == "cls":\n                pooled = hidden[:, 0]\n            elif prep.padding:\n                m = mask.unsqueeze(-1).type_as(hidden)\n                pooled = (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)\n            else:\n                pooled = hidden.mean(1)\n            if normalize:\n                pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)\n            out[idx] = pooled.float().cpu().numpy()\n    return out\n\n\ndef _dense_features(data: dict, cfg: ExperimentConfig):\n    from transformers import AutoModel, AutoTokenizer\n    from transformers.utils import logging as hf_logging\n\n    hf_logging.set_verbosity_error()\n    hf_logging.disable_progress_bar()\n    spec = DENSE_MODELS[cfg.embedding_type]\n    device = get_device()\n    tokenizer = AutoTokenizer.from_pretrained(spec["model_id"])\n    model = AutoModel.from_pretrained(spec["model_id"]).to(device)\n    max_length = resolve_max_length(data["train_text"], tokenizer, cfg.preprocess, data["max_len_cap"])\n    print(f"[features] {spec[\'model_id\']} on {device}, max_length={max_length}, "\n          f"pooling={spec[\'pooling\']}, tokenization={cfg.preprocess.tokenization}, "\n          f"padding/masking={cfg.preprocess.padding}")\n    args = (tokenizer, model, device, cfg.preprocess, max_length, spec["pooling"], spec["normalize"])\n    x_train = _encode(data["train_text"], *args)\n    x_test = _encode(data["test_text"], *args)\n    scaler = StandardScaler().fit(x_train)\n    pca = PCA(n_components=PCA_DIM, random_state=SEED).fit(scaler.transform(x_train))\n    z_train = pca.transform(scaler.transform(x_train)).astype(np.float32)\n    z_test = pca.transform(scaler.transform(x_test)).astype(np.float32)\n    unk_rate = None\n    if not cfg.preprocess.tokenization:\n        vocab = tokenizer.get_vocab()\n        words = [w for t in data["train_text"] for w in t.split()]\n        unk_rate = round(float(np.mean([w not in vocab for w in words])), 4)\n    meta = {\n        "model_id": spec["model_id"],\n        "pooling": spec["pooling"],\n        "l2_normalized": spec["normalize"],\n        "max_length": max_length,\n        "unk_word_rate": unk_rate,\n        "projection": f"StandardScaler + PCA({PCA_DIM})",\n        "explained_variance_30d": round(float(pca.explained_variance_ratio_.sum()), 4),\n    }\n    del model\n    if device.type == "mps":\n        import torch\n\n        torch.mps.empty_cache()\n    return x_train, x_test, z_train, z_test, meta\n\n\n# ---------------------------------------------------------------------------\n# Word2Vec (static word vectors + mean-pooled document features)\n# ---------------------------------------------------------------------------\ndef _mean_pool_word2vec(texts, kv) -> np.ndarray:\n    dim = kv.vector_size\n    out = np.zeros((len(texts), dim), dtype=np.float32)\n    for i, text in enumerate(texts):\n        vecs = [kv[w] for w in str(text).split() if w in kv]\n        if vecs:\n            out[i] = np.mean(vecs, axis=0)\n    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)\n\n\ndef train_or_load_word2vec(train_texts, folder: Path, seed: int = SEED):\n    """Train skip-gram Word2Vec on train texts, or load WORD2VEC_KV_PATH / cached kv."""\n    from modules.word2vec_utils import load_or_train_kv\n\n    return load_or_train_kv(train_texts, folder, seed)\n\n\ndef _word2vec_features(data: dict, cfg: ExperimentConfig, folder: Path):\n    kv = train_or_load_word2vec(data["train_text"], folder, SEED)\n    x_train = _mean_pool_word2vec(data["train_text"], kv)\n    x_test = _mean_pool_word2vec(data["test_text"], kv)\n    scaler = StandardScaler().fit(x_train)\n    z_train_full = np.nan_to_num(scaler.transform(x_train), nan=0.0, posinf=0.0, neginf=0.0)\n    z_test_full = np.nan_to_num(scaler.transform(x_test), nan=0.0, posinf=0.0, neginf=0.0)\n    pca = PCA(n_components=PCA_DIM, random_state=SEED).fit(z_train_full)\n    z_train = pca.transform(z_train_full).astype(np.float32)\n    z_test = pca.transform(z_test_full).astype(np.float32)\n    x_train = np.nan_to_num(x_train, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)\n    x_test = np.nan_to_num(x_test, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)\n    covered = sum(1 for t in data["train_text"] for w in t.split() if w in kv)\n    total = sum(len(t.split()) for t in data["train_text"]) or 1\n    meta = {\n        "model_id": "word2vec-skipgram-train",\n        "pooling": "mean",\n        "vector_size": int(kv.vector_size),\n        "vocab_size": int(len(kv)),\n        "token_coverage": round(covered / total, 4),\n        "kv_path": str(folder / "word2vec.kv.npz"),\n        "projection": f"StandardScaler + PCA({PCA_DIM})",\n        "explained_variance_30d": round(float(pca.explained_variance_ratio_.sum()), 4),\n    }\n    return x_train, x_test, z_train, z_test, meta\n\n\ndef load_word2vec_kv(dataset_name: str, prep_tag: str, feature_root: Path = FEATURE_DIR):\n    """Load the cached KeyedVectors for a dataset/preprocess tag (trains if missing)."""\n    from modules.word2vec_utils import SimpleKeyedVectors\n\n    folder = cache_dir_for(dataset_name, "word2vec", prep_tag, feature_root)\n    kv_path = folder / "word2vec.kv.npz"\n    if kv_path.exists():\n        return SimpleKeyedVectors.load(kv_path)\n    raise FileNotFoundError(\n        f"No Word2Vec cache at {kv_path}. Call extract_and_save_features(..., embedding_type=\'word2vec\') first."\n    )\n\n\n# ---------------------------------------------------------------------------\n# Public entry point\n# ---------------------------------------------------------------------------\ndef extract_and_save_features(dataset_dict: dict, exp_cfg: ExperimentConfig,\n                              feature_root: Path = FEATURE_DIR) -> dict:\n    """\n    Build (or load from cache) one embedding for one dataset and preprocessing tag.\n\n    Returns a dict with X_train / X_test (full representation, sparse or dense),\n    X_train_pca30 / X_test_pca30, is_sparse, dim, cache_dir, extract_time_sec, meta.\n    """\n    emb = exp_cfg.embedding_type\n    known = set(SPARSE_TYPES) | set(DENSE_MODELS) | {"word2vec"}\n    if emb not in known:\n        raise ValueError(f"Unknown embedding_type \'{emb}\'")\n    tag = exp_cfg.preprocess.tag()\n    if dataset_dict.get("preprocess_tag") not in (None, tag):\n        raise ValueError(f"dataset was cleaned with {dataset_dict[\'preprocess_tag\']} but config asks for {tag}")\n    folder = cache_dir_for(exp_cfg.dataset_name, emb, tag, feature_root)\n    is_sparse = emb in SPARSE_TYPES\n    y_train, y_test = dataset_dict["train_label"], dataset_dict["test_label"]\n\n    cached = _load_cached(folder, is_sparse, y_train, y_test)\n    if cached is not None and (emb != "word2vec" or (folder / "word2vec.kv.npz").exists()):\n        out = {**cached, "is_sparse": is_sparse, "cache_dir": folder,\n               "extract_time_sec": cached["meta"].get("extract_time_sec", 0.0), "from_cache": True}\n        print(f"[features] loaded {emb}_{tag} from {folder}: train {out[\'X_train\'].shape}, "\n              f"test {out[\'X_test\'].shape}, 30-d {out[\'X_train_pca30\'].shape}")\n    else:\n        start = time.perf_counter()\n        if is_sparse:\n            x_train, x_test, z_train, z_test, meta = _sparse_features(dataset_dict, exp_cfg)\n        elif emb == "word2vec":\n            x_train, x_test, z_train, z_test, meta = _word2vec_features(dataset_dict, exp_cfg, folder)\n        else:\n            x_train, x_test, z_train, z_test, meta = _dense_features(dataset_dict, exp_cfg)\n        elapsed = time.perf_counter() - start\n        meta.update({\n            "dataset": exp_cfg.dataset_name,\n            "embedding_type": emb,\n            "preprocess": tag,\n            "n_train": int(len(y_train)),\n            "n_test": int(len(y_test)),\n            "dim": int(x_train.shape[1]),\n            "extract_time_sec": round(elapsed, 3),\n        })\n        arrays = {"X_train_pca30": z_train, "X_test_pca30": z_test, "y_train": y_train, "y_test": y_test}\n        sparse_arrays = {}\n        if is_sparse:\n            sparse_arrays = {"X_train_sparse": x_train, "X_test_sparse": x_test}\n        else:\n            arrays.update({"X_train": x_train, "X_test": x_test})\n        _save(folder, arrays, sparse_arrays, meta, exp_cfg.save_format)\n        out = {"X_train": x_train, "X_test": x_test, "X_train_pca30": z_train, "X_test_pca30": z_test,\n               "meta": meta, "is_sparse": is_sparse, "cache_dir": folder,\n               "extract_time_sec": elapsed, "from_cache": False}\n        print(f"[features] {emb}_{tag}: train {x_train.shape}, test {x_test.shape}, "\n              f"30-d {z_train.shape} in {elapsed:.1f}s -> {folder}")\n    out["dim"] = int(out["X_train"].shape[1])\n    out["embedding"] = emb\n    return out\n\n\ndef list_feature_files(feature_root: Path = FEATURE_DIR):\n    """Rows (dataset, folder, file, size_mb) for every saved .npy / .npz / .h5 file."""\n    rows = []\n    for path in sorted(Path(feature_root).rglob("*")):\n        if path.suffix in (".npy", ".npz", ".h5"):\n            rel = path.relative_to(feature_root)\n            rows.append({\n                "dataset": rel.parts[0],\n                "folder": rel.parts[1] if len(rel.parts) > 2 else "",\n                "file": path.name,\n                "size_mb": round(path.stat().st_size / 1e6, 3),\n            })\n    return rows\n', 'utils.py': '"""Small helpers shared by every module: seeding and device selection."""\n\nfrom __future__ import annotations\n\nimport os\nimport random\n\nimport numpy as np\n\nfrom modules.config import SEED\n\n\ndef seed_everything(seed: int = SEED) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    os.environ.setdefault("PYTHONHASHSEED", str(seed))\n    try:\n        import torch\n\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n    except ImportError:\n        pass\n\n\ndef get_device(prefer_gpu: bool = True):\n    """CUDA on Colab, Apple MPS locally, CPU otherwise."""\n    import torch\n\n    if prefer_gpu and torch.cuda.is_available():\n        return torch.device("cuda")\n    if prefer_gpu and getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n', 'word2vec_utils.py': '"""\nLightweight skip-gram Word2Vec (no gensim) for environments where gensim wheels\nare unavailable. Provides a KeyedVectors-like object used by feature extraction\nand the DL / SetFit adapters.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom modules.config import SEED, WORD2VEC\nfrom modules.data_loader import quick_mode\n\n_WEIGHT_CLIP = 1.0\n\n\nclass SimpleKeyedVectors:\n    """Minimal gensim.KeyedVectors-compatible surface."""\n\n    def __init__(self, vectors: np.ndarray, index_to_key: list[str]):\n        self.vectors = np.asarray(vectors, dtype=np.float32)\n        self.index_to_key = list(index_to_key)\n        self.key_to_index = {w: i for i, w in enumerate(self.index_to_key)}\n        self.vector_size = int(self.vectors.shape[1])\n\n    def __contains__(self, key: str) -> bool:\n        return key in self.key_to_index\n\n    def __len__(self) -> int:\n        return len(self.index_to_key)\n\n    def __getitem__(self, key: str) -> np.ndarray:\n        return self.vectors[self.key_to_index[key]]\n\n    def is_finite(self) -> bool:\n        return bool(np.isfinite(self.vectors).all())\n\n    def save(self, path: str | Path) -> None:\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        np.savez_compressed(path, vectors=self.vectors, words=np.array(self.index_to_key, dtype=object))\n        # also write word2vec text for debugging / optional ST loaders\n        txt = path.with_suffix(".txt") if path.suffix == ".npz" else Path(str(path) + ".txt")\n        # Prefer word2vec.kv.txt next to word2vec.kv.npz\n        if path.name.endswith(".kv.npz"):\n            txt = path.with_name(path.name.replace(".kv.npz", ".kv.txt"))\n        with txt.open("w", encoding="utf-8") as f:\n            f.write(f"{len(self)} {self.vector_size}\\n")\n            for w, i in self.key_to_index.items():\n                vals = " ".join(f"{x:.6f}" for x in self.vectors[i])\n                f.write(f"{w} {vals}\\n")\n\n    def save_word2vec_format(self, path: str | Path) -> None:\n        self.save(Path(path).with_suffix(".kv.npz") if not str(path).endswith(".npz") else path)\n\n    @classmethod\n    def load(cls, path: str | Path, mmap: str | None = None) -> "SimpleKeyedVectors":\n        path = Path(path)\n        data = np.load(path, allow_pickle=True)\n        words = [str(w) for w in data["words"].tolist()]\n        return cls(data["vectors"], words)\n\n\ndef _build_vocab(sentences: list[list[str]], min_count: int) -> list[str]:\n    counts = Counter(w for s in sentences for w in s)\n    return [w for w, c in counts.most_common() if c >= min_count]\n\n\ndef _sigmoid(x: np.ndarray) -> np.ndarray:\n    return 1.0 / (1.0 + np.exp(-np.clip(x, -20.0, 20.0)))\n\n\ndef _sg_update(w_in: np.ndarray, w_out: np.ndarray, center: int, targets: np.ndarray,\n               labels: np.ndarray, lr: float) -> None:\n    """Vectorized skip-gram NEG update for one center vs many target ids."""\n    if targets.size == 0:\n        return\n    # Copy so in/out updates do not alias.\n    v_c = w_in[center].copy()\n    w_t = w_out[targets]\n    scores = w_t @ v_c\n    pred = _sigmoid(scores)\n    g = (lr * (labels.astype(np.float32) - pred)).astype(np.float32)[:, None]\n    w_in[center] += (g * w_t).sum(axis=0)\n    w_out[targets] += g * v_c[None, :]\n    np.clip(w_in[center], -_WEIGHT_CLIP, _WEIGHT_CLIP, out=w_in[center])\n    np.clip(w_out[targets], -_WEIGHT_CLIP, _WEIGHT_CLIP, out=w_out[targets])\n\n\ndef train_skipgram(train_texts, seed: int = SEED) -> SimpleKeyedVectors:\n    """Train a small skip-gram model with negative sampling (vectorized numpy)."""\n    rng = np.random.default_rng(seed)\n    sentences = [str(t).split() for t in train_texts]\n    if quick_mode():\n        sentences = sentences[:800]\n    dim = 100 if quick_mode() else WORD2VEC["vector_size"]\n    window = WORD2VEC["window"]\n    min_count = WORD2VEC["min_count"]\n    epochs = 1 if quick_mode() else WORD2VEC["epochs"]\n    neg = 5\n    vocab = _build_vocab(sentences, min_count)\n    if not vocab:\n        vocab = ["unk"]\n    word2id = {w: i for i, w in enumerate(vocab)}\n    n = len(vocab)\n    w_in = rng.normal(0, 0.01, (n, dim)).astype(np.float32)\n    w_out = rng.normal(0, 0.01, (n, dim)).astype(np.float32)\n    lr = 0.025\n\n    freqs = np.ones(n, dtype=np.float64)\n    counts = Counter(w for s in sentences for w in s if w in word2id)\n    for w, c in counts.items():\n        freqs[word2id[w]] = c\n    noise = freqs ** 0.75\n    noise /= noise.sum()\n\n    print(f"[features] Word2Vec skip-gram: vocab={n}, dim={dim}, epochs={epochs}, docs={len(sentences)}")\n    for epoch in range(epochs):\n        order = rng.permutation(len(sentences))\n        for si in order:\n            ids = [word2id[w] for w in sentences[si] if w in word2id]\n            if len(ids) < 2:\n                continue\n            for i, center in enumerate(ids):\n                left = max(0, i - window)\n                right = min(len(ids), i + window + 1)\n                ctx = np.array([ids[j] for j in range(left, right) if j != i], dtype=np.int64)\n                if ctx.size == 0:\n                    continue\n                # positives\n                _sg_update(w_in, w_out, center, ctx, np.ones(ctx.size, dtype=np.float32), lr)\n                # negatives (shared draw per center position)\n                neg_ids = rng.choice(n, size=neg * ctx.size, p=noise).astype(np.int64)\n                # avoid accidental identity with center\n                neg_ids[neg_ids == center] = (center + 1) % n\n                _sg_update(w_in, w_out, center, neg_ids, np.zeros(neg_ids.size, dtype=np.float32), lr)\n        lr = max(lr * 0.95, 0.001)\n        np.clip(w_in, -_WEIGHT_CLIP, _WEIGHT_CLIP, out=w_in)\n        np.clip(w_out, -_WEIGHT_CLIP, _WEIGHT_CLIP, out=w_out)\n        print(f"[features] Word2Vec epoch {epoch + 1}/{epochs} done")\n\n    np.nan_to_num(w_in, copy=False, nan=0.0, posinf=0.0, neginf=0.0)\n    np.clip(w_in, -_WEIGHT_CLIP, _WEIGHT_CLIP, out=w_in)\n    return SimpleKeyedVectors(w_in, vocab)\n\n\ndef load_or_train_kv(train_texts, folder: Path, seed: int = SEED) -> SimpleKeyedVectors:\n    import os\n\n    folder = Path(folder)\n    folder.mkdir(parents=True, exist_ok=True)\n    kv_path = folder / "word2vec.kv.npz"\n    pretrained = os.environ.get(WORD2VEC["pretrained_env"], "").strip()\n    if pretrained and Path(pretrained).exists():\n        print(f"[features] loading Word2Vec from {pretrained}")\n        kv = SimpleKeyedVectors.load(pretrained)\n        if not kv.is_finite():\n            raise ValueError(f"Pretrained Word2Vec at {pretrained} contains NaN/Inf")\n        kv.save(kv_path)\n        return kv\n    if kv_path.exists():\n        kv = SimpleKeyedVectors.load(kv_path)\n        if kv.is_finite():\n            return kv\n        print(f"[features] cached Word2Vec at {kv_path} has NaN/Inf; deleting and retraining")\n        kv_path.unlink(missing_ok=True)\n        txt = folder / "word2vec.kv.txt"\n        txt.unlink(missing_ok=True)\n    print("[features] training lightweight skip-gram Word2Vec (no gensim)...")\n    kv = train_skipgram(train_texts, seed)\n    if not kv.is_finite():\n        raise RuntimeError("Word2Vec training produced non-finite vectors after sanitization")\n    kv.save(kv_path)\n    print(f"[features] Word2Vec vocab={len(kv)}, dim={kv.vector_size}")\n    return kv\n'}

here = Path.cwd()
if (here / "modules" / "__init__.py").exists():
    ROOT = here
elif (here.parent / "modules" / "__init__.py").exists():
    ROOT = here.parent
else:
    ROOT = here
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=False)
        if (here / "repo" / "modules" / "__init__.py").exists():
            ROOT = here / "repo"
    if not (ROOT / "modules" / "__init__.py").exists():
        (ROOT / "modules").mkdir(exist_ok=True)
        for filename, text in MODULE_SOURCES.items():
            (ROOT / "modules" / filename).write_text(text)
        print(f"wrote {len(MODULE_SOURCES)} module files to {ROOT / 'modules'}")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("project root:", ROOT)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

from modules import (DATASET_ORDER, DATASET_REGISTRY, FEATURE_DIR, REPORT_DIR, FIG_DIR, PreprocessConfig,
                     clean_text, seed_everything, get_device, run_all_eda, run_preprocessing_ablation,
                     run_feature_extraction, list_feature_files, load_h5_features, run_main_comparison,
                     run_deep_learning, plot_suite_results, error_analysis, negation_analysis, build_master_table)
from modules import evaluation as ev

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
seed_everything(42)
print("device:", get_device())
for name in DATASET_ORDER:
    cfg = DATASET_REGISTRY[name]
    print(f"{cfg['title']:16s} https://huggingface.co/datasets/{cfg['hf_id']}")

## 1. Exploratory data analysis

| Dataset (Hugging Face) | Task | Classes | Split used | Role |
| --- | --- | --- | --- | --- |
| `mteb/banking77` | online-banking customer intent | 77 | 9,993 / 3,076 (official) | fine-grained intents that share nouns and differ by function words / negation |

We report class balance, word-count statistics, vocabulary size with and without stop words, and top-15 words. The 95th percentile length is the padding cut-off for DistilBERT and BiLSTM.

In [ ]:
eda = run_all_eda(DATASET_ORDER, show=True)
display(eda.set_index("dataset").T)

## 2. Boolean preprocessing configuration and ablation

`PreprocessConfig(stop_word_removal, tokenization, padding, negation_marking)` is passed to every step; its tag (e.g. `sw0_tok1_pad1_neg0`) names the feature cache folder.

| Switch | `False` | `True` |
| --- | --- | --- |
| `stop_word_removal` | keep every word | drop scikit-learn `ENGLISH_STOP_WORDS` |
| `tokenization` | whitespace unigrams / whole-word lookup | bigrams for TF-IDF; WordPiece for DistilBERT |
| `padding` | truncate to 10 tokens; no masking | pad/truncate to 95th-percentile length with masks |
| `negation_marking` | unchanged | join negation cues with the next word (`not arrived` -> `not_arrived`) |

Default main configuration: `sw0_tok1_pad1_neg0`. Ablation flips **one switch at a time** for TF-IDF + LogReg, DistilBERT + LogReg, and Word2Vec + BiLSTM.

In [ ]:
example = "I'm still waiting for my card, it has NOT arrived from the bank yet!"
for cfg in [PreprocessConfig(False, True, True, False), PreprocessConfig(True, True, True, False),
            PreprocessConfig(False, True, True, True)]:
    print(f"{cfg.tag()}: {clean_text(example, cfg.stop_word_removal, True, cfg.negation_marking)!r}")

In [ ]:
ablation = {}
for name in DATASET_ORDER:
    runs, delta = run_preprocessing_ablation(name)
    ablation[name] = delta
    display(delta[["dataset", "model", "flag", "f1_false", "f1_true", "delta_f1", "note"]].round(4))
    ev.plot_ablation_heatmap(name, delta, show=True)

## 3. Feature extraction (saved as `.npy` and `.h5`)

| Key | Family | Representation |
| --- | --- | --- |
| `tfidf` | traditional | sublinear TF-IDF, uni+bigrams, `min_df=2` (sparse `.npz`) |
| `word2vec` | static | skip-gram on train texts, mean-pooled 300-d (+ `word2vec.kv.npz` for BiLSTM) |
| `distilbert` | modern | frozen `distilbert-base-uncased`, attention-masked mean pooling, 768-d |

Fit on the training split only. Each folder `features/banking77/<embedding>_<tag>/` holds dense `.npy` or sparse `.npz`, optional 30-d projections, labels, `features.h5`, and `meta.json`.

In [ ]:
feature_table = pd.concat([run_feature_extraction(name) for name in DATASET_ORDER], ignore_index=True)
display(feature_table)

In [ ]:
files = pd.DataFrame(list_feature_files(FEATURE_DIR))
summary = files.groupby(["dataset", "folder"]).agg(files=("file", "count"), size_mb=("size_mb", "sum")).round(2)
display(summary)
print(f"{len(files)} feature files, {files['size_mb'].sum():.1f} MB in {FEATURE_DIR}")

folder = FEATURE_DIR / "banking77" / "distilbert_sw0_tok1_pad1_neg0"
h5 = load_h5_features(folder / "features.h5")
print("h5 keys:", sorted(k for k in h5 if k != "meta"))
print("X_train.npy == features.h5['X_train']:", np.array_equal(np.load(folder / "X_train.npy"), h5["X_train"]))

## 4. Classifiers on the saved features

| Classifier | Sparse input (TF-IDF) | Dense input (embeddings) |
| --- | --- | --- |
| Naive Bayes | `MultinomialNB(alpha=0.5)` | `GaussianNB()` |
| Logistic Regression | `LogisticRegression(C=2.0, balanced)` | StandardScaler + same |
| Linear SVM | calibrated `LinearSVC(C=1.0)` (Platt) | L2 Normalizer + same |

Metrics: accuracy, macro precision / recall / F1, multiclass Brier score, ECE.

In [ ]:
show_cols = ["method", "input_dim", "accuracy", "f1", "brier_score", "ece", "train_time_sec", "infer_time_sec"]
main_tables = {}
for name in DATASET_ORDER:
    main_tables[name] = run_main_comparison(name)
    display(main_tables[name][show_cols].round(4))

## 5. Bonus deep-learning pipeline (Word2Vec + BiLSTM)

A bidirectional LSTM over the same skip-gram Word2Vec table used for mean-pooled document vectors. Packing / masking follows the `padding` preprocess switch. Early stopping uses a stratified 10% validation split of the training set.

In [ ]:
dl_tables = {}
for name in DATASET_ORDER:
    dl_tables[name] = run_deep_learning(name)
    display(dl_tables[name][["dataset", "method", "accuracy", "f1", "brier_score", "ece",
                             "train_time_sec", "infer_time_sec", "notes"]].round(4))
    loss_png = FIG_DIR / name / "bilstm_loss.png"
    if loss_png.exists():
        display(Image(filename=str(loss_png)))

## 6. Results, calibration and error analysis

Accuracy / Macro-F1 bars, Brier vs Macro-F1, reliability diagram, ablation heat map, confusion matrix of the best model, frequent confusions, and accuracy on negated vs non-negated test queries.

In [ ]:
results = {}
for name in DATASET_ORDER:
    print("=" * 100)
    print(DATASET_REGISTRY[name]["title"])
    results[name] = ev.save_comparison(name)
    display(results[name][["method", "family", "accuracy", "precision", "recall", "f1", "brier_score", "ece"]].round(4))
    plot_suite_results(name, results[name], ablation.get(name), show=True)
    display(error_analysis(name)[["true", "predicted", "count", "share_of_true_class", "example_1"]])
    display(negation_analysis(name).round(4))

In [ ]:
master = build_master_table(DATASET_ORDER, show=True)
pivot = pd.read_csv(REPORT_DIR / "master_f1_pivot.csv", index_col=0)
display((pivot * 100).round(2))
best = master.loc[master.groupby("dataset")["f1"].idxmax(), ["dataset", "method", "accuracy", "f1", "brier_score"]]
display(best.round(4))

In [ ]:
print("Deliverables written by this notebook:")
for path in sorted(REPORT_DIR.glob("*.csv")):
    print("  ", path.relative_to(ROOT))
print(f"   {len(list(FIG_DIR.rglob('*.png')))} figures under {FIG_DIR.relative_to(ROOT)}/")
print(f"   {len(list_feature_files(FEATURE_DIR))} feature files (.npy / .npz / .h5) under {FEATURE_DIR.relative_to(ROOT)}/")

## 7. Conclusions

Numbers are Macro-F1 on the Banking77 test set from this notebook run (single seed 42).

1. **Fine-grained intent classification stresses lexical baselines**: TF-IDF remains strong, but frozen DistilBERT and sequence modelling test whether representation geometry separates overlapping intents.
2. **Preprocessing is representation-dependent**: stop-word removal, tokenization, padding, and negation marking are ablated one at a time on TF-IDF + LogReg, DistilBERT + LogReg, and Word2Vec + BiLSTM.
3. **Mean-pooled Word2Vec vs BiLSTM** isolates the value of order for short banking queries under a shared static embedding table.
4. **Macro-F1 and calibration** (Brier, ECE) must be read together on a 77-way problem with class imbalance.